In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2 diptest>=0.11.0
!pip install -q flash-linear-attention==0.4.2 --no-build-isolation
!pip install -q transformers==5.15.1


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import contextlib
import logging
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    prefix_cache=None,
    h_override: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
        prefix_cache: optional pre-computed prefix cache (K/V for positions 0..S-2).
        h_override: if given, used as the hidden state of the last position at
            layer_idx INSTEAD of extracting it from a full forward. The prefix
            cache is still built from input_ids[:, :-1] (context unchanged); the
            caller is responsible for h_override being consistent with
            input_ids (e.g. h_override from the same sequence). Shapes: [1, 1, D],
            [1, D] or [D]. Device must match input_ids.device (no auto-move).
            dtype is cast to the model dtype. The tensor is cloned and detached
            (fresh leaf; the caller's tensor is never mutated).
    Returns:
        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).
    Interaction with compute_hidden_jacobian:
        compute_jacobian_for_tokens takes `h_override`; compute_hidden_jacobian
        takes `h`. Both mean "use this h instead of computing it from the
        forward", but the names differ for historical reasons (compute_hidden_
        jacobian already had `h`). With h_override=None the behavior is
        bit-identical to pre-v6.0.
    """
    model.eval()
    S = input_ids.shape[1]
    # Constraint: single sequence, no padding. The replay phase builds causal
    # masks from the cache length with attention_mask=None, so any padding
    # token in the prefix would be silently treated as valid context.
    if attention_mask is not None and not bool(attention_mask.all()):
        raise ValueError(
            "compute_jacobian_for_tokens: only unpadded single sequences are "
            "supported (the replay phase cannot honor a padding mask)."
        )
    if prefix_cache is None:
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h_override is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
        h = h.to(next(model.parameters()).dtype).detach().requires_grad_(True)
    else:
        h = _normalize_h_override(h_override, model, input_ids)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _normalize_h_override(
    h_override: torch.Tensor, model, input_ids: torch.Tensor
) -> torch.Tensor:
    """Validate + normalize h_override to a detached [1, 1, D] leaf.
    Checks (in order, before any compute): shape reducible to [1, 1, D],
    D == model hidden_size, device == input_ids.device. dtype is cast to the
    model dtype. The tensor is cloned+detached (fresh leaf; caller's tensor
    never mutated, no parasitic CopyBackwards nodes).
    """
    if h_override.dim() not in (1, 2, 3) or h_override.numel() == 0:
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    if any(s != 1 for s in h_override.shape[:-1]):
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    h = h_override.reshape(1, 1, -1)
    # D lookup: flat CausalLM has config.hidden_size; the multimodal wrapper
    # (Qwen3_5ForConditionalGeneration) exposes it on config.text_config only
    # (the outer config is a heterogeneity spec without hidden_size).
    D_expected = None
    cfg = model.config
    for cand in (getattr(cfg, "text_config", None), cfg):
        if cand is not None and hasattr(cand, "hidden_size"):
            D_expected = cand.hidden_size
            break
    if D_expected is None:
        raise ValueError(
            "h_override: cannot resolve model hidden_size "
            "(neither config nor config.text_config has it)"
        )
    D = h.shape[-1]
    if D != D_expected:
        raise ValueError(
            f"h_override: D={D} does not match model hidden_size={D_expected}"
        )
    if h.device != input_ids.device:
        raise ValueError(
            f"h_override: device {h.device} does not match "
            f"input_ids.device {input_ids.device} (no auto-move)"
        )
    model_dtype = next(model.parameters()).dtype
    # clone first, then cast: detach/clone act on the caller's values before
    # any cast, and `to(same dtype)` is a no-op, so the typical fp32 case
    # allocates exactly once (audit docs/changed_review.md, item 3).
    return h.detach().clone().to(model_dtype).requires_grad_(True)
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def last_position_logits(
    model,
    h: torch.Tensor,
    S: int,
    layer_idx: int,
    prefix_cache: DynamicCache,
) -> torch.Tensor:
    """Decode h (residual at position S-1, output of layer layer_idx) through the
    remaining layers + final norm + lm_head against a prefix cache over positions
    0..S-2 and return logits [1, V].
    This is EXACTLY the path compute_jacobian_for_tokens differentiates through,
    so with prefix_cache = _run_prefix_pass(m, ids, attn) and h =
    hidden_states[layer_idx+1][:, -1:, :] it reproduces the full-forward
    logits[:, -1, :] (KL ~ 1e-7, verified). Passing the cache of a FULL forward
    (all S positions) instead double-counts the last token — the bug this helper
    exists to prevent (audit 2026-11-05, E-11a/12a/12b)."""
    return _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    Must copy every per-layer state field, not just the obvious tensors. Both
    layer types swallow unknown constructor kwargs, so passing
    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the
    state and yields an empty cache. We therefore assign attributes directly:
    * DynamicLayer (full attention): keys / values / is_initialized / dtype /
      device. Without this, ``update()`` lazy-initialises an empty cache and
      each layer attends only to the current token.
    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag
      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the
      new token as the first in the sequence and drop all history.
    All cloned state tensors are .detach()-ed: the prefix cache is pure data
    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes
    in the enable_grad decode. The transformers 5.15 in-place cache updates
    (update_recurrent_state's copy_, causal_conv1d_update's state update —
    cudagraph static-address optimisation) are NOT fixed by the detach —
    the in-place copy_ lands on the kernel's saved input regardless. Those
    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the
    read object and the in-place-written object are different objects).
    Extra unknown attributes are deep-copied as a forward-compat safety net.
    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried
    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:
    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``
    is derived from ``keys.shape`` (not an object field), so the post-clone
    assert below passes on both paths — on a version that tracks sequence
    length in cache state it turns a silent Jacobian corruption into a loud
    error.
    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1
    (tensor values are not shared). _deep_clone_cache exists for (a) detached
    tensors (no spurious CopyBackwards in the enable_grad decode) and
    (b) robustness to per-layer attribute renames — not because deepcopy
    shares storage. Neither clone fixes the in-place write into a graph-saved
    node; that is _clone_gdn_states_on_entry's job (see its docstring).
    """
    import copy as _copy
    def _cl(t):
        return t.clone().detach() if t is not None else None
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            # Deliberately NOT a vars()-walk: full-attention state fields are
            # stable across versions, and a rename would fail LOUD
            # (AttributeError), not silently corrupt the decode. The GDN
            # branch below IS generic, because GDN state fields have
            # demonstrably drifted between versions/backends.
            nl = DynamicLayer()
            nl.keys = _cl(layer.keys)
            nl.values = _cl(layer.values)
            nl.is_initialized = layer.is_initialized
            nl.dtype = getattr(layer, "dtype", None)
            nl.device = getattr(layer, "device", None)
            for k, v in vars(layer).items():
                if k not in vars(nl):
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
        else:
            # GDN (LinearAttentionLayer). Clone every state field dynamically
            # rather than hardcoding transformers-5.15.1 attribute names
            # (number_of_states, conv_states, ...): other transformers versions
            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some
            # differently, and a hardcoded name crashes with AttributeError.
            # Tensors (or dicts of tensors) are clone+detached; scalars and
            # flag-dicts are deep-copied. This is the generic form of the old
            # field-by-field copy and behaves identically on 5.15.1.
            nl = LinearAttentionLayer()
            for k, v in vars(layer).items():
                if isinstance(v, torch.Tensor):
                    setattr(nl, k, _cl(v))
                elif isinstance(v, dict) and v and all(
                    isinstance(x, torch.Tensor) for x in v.values()
                ):
                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})
                else:
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):
    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;
    # list-valued attrs are skipped to avoid aliasing the original.
    for k, v in vars(cache).items():
        if k == "layers" or isinstance(v, list):
            continue
        setattr(new_cache, k, _copy.deepcopy(v))
    # Guard: the clone must report the same sequence length (loud, not
    # silent, on a version that tracks it in cache state). A bare `assert`
    # disappears under `python -O`, turning a fail-loud check into a silent
    # one — the class of bug this guard exists to catch (audit4 Q12).
    seq_new, seq_old = new_cache.get_seq_length(), cache.get_seq_length()
    if seq_new != seq_old:
        raise RuntimeError(
            f"cache clone lost the sequence length: {seq_new} != {seq_old} "
            "(transformers cache layout changed? the clone is not a clone)")
    return new_cache
@contextlib.contextmanager
def _clone_gdn_states_on_entry():
    """Clone GDN kernel state inputs at entry: read object != written object.
    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as
    free variables of its own module (modeling_qwen3_5), so patching the
    module attributes works — verified on 5.15.1 (calls at modeling L457/
    L506). On
    a version where GDN forward binds the kernels by a different route
    (an alias imported into another module), the patch silently no-ops and
    the in-place autograd error below resurfaces LOUD — that is the
    detection signal; there is no silent degradation.
    The recurrent loop saves the read object (version 0) for backward.
    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on
    the very same object (5.15 cudagraph static-address optimisation) →
    version 0→1 → "modified by an inplace operation" in any backward through
    the decode (create_graph not required; the clone is also a no-op safety
    net for create_graph=True graph retention). Cloning at kernel entry makes
    the loop's saved input the fresh clone; the in-place write lands on the
    cache's original (a different object) → no in-place on a saved node.
    Re-entrancy: nesting is a NO-OP (counter below) — the outer context's
    wrappers are already active and cloning is idempotent (clone-of-clone
    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch
    window is user error and is NOT protected (the outer exit restores its
    captured orig, discarding the overwrite).
    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per
    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5
    with enable_grad=True the in-place autograd error is NOT protected).
    """
    global _GDN_PATCH_DEPTH
    try:
        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
    except Exception:
        if not getattr(_clone_gdn_states_on_entry, "_warned", False):
            logging.getLogger(__name__).warning(
                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "
                "importable). If the model is Qwen3.5 with enable_grad=True, "
                "in-place autograd errors may occur.")
            _clone_gdn_states_on_entry._warned = True
        yield  # non-Qwen3.5 build: nothing to patch
        return
    _GDN_PATCH_DEPTH += 1
    if _GDN_PATCH_DEPTH > 1:
        # Nested: outer context already patched; wrappers chain, and the
        # clone is idempotent, so nothing to do here.
        yield
        _GDN_PATCH_DEPTH -= 1
        return
    orig_rec = mq.torch_recurrent_gated_delta_rule
    orig_conv = mq.causal_conv1d_update
    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,
            use_qk_l2norm_in_kernel=False, **kw):
        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1
        # (and may be positional on other versions, hence *args); clone the
        # bound state, forward everything else untouched. A kernel whose
        # signature drops these names falls through to **kw and fails loud.
        if initial_state is not None:
            initial_state = initial_state.clone()
        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,
                        output_final_state=output_final_state,
                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)
    def conv(h, cs, *args, **kw):
        # (cs, weight, [bias, activation], ...) — positional tail is
        # version-dependent; forward everything we don't own.
        if cs is not None:
            cs = cs.clone()
        return orig_conv(h, cs, *args, **kw)
    mq.torch_recurrent_gated_delta_rule = rec
    mq.causal_conv1d_update = conv
    try:
        yield
    finally:
        mq.torch_recurrent_gated_delta_rule = orig_rec
        mq.causal_conv1d_update = orig_conv
        _GDN_PATCH_DEPTH -= 1
_GDN_PATCH_DEPTH = 0
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model
    text_model = _lm_module(model)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    layer_types = _layer_types(model)
    n_layers = _n_layers(model)
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via ONE batched backward (is_grads_batched).
    Replaces the per-row autograd.grad loop (K backwards through the
    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)
    gives all K rows at once, 3-15x faster on T4 (review point 5).
    Verified identical to the loop on synthetic linear + nonlinear graphs;
    on the real GDN decoder tail this is an EMPIRICAL property of the
    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal
    conv1d update under vmap-style batched backward) — there is no
    per-row fallback, so a transformers upgrade that breaks this shows up
    as a runtime error, not silent degradation.
    """
    K = len(token_set)
    return torch.autograd.grad(
        outputs=logits[0, token_set],
        inputs=h,
        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),
        is_grads_batched=True,
        retain_graph=False,
    )[0][:, 0, 0, :]
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].
    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the
    # final-token logits), so collapse a [1, S, D] to its last position first.
    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity
    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU
    # (previously misattributed to an "XLA phantom dot" shape bug).
    if h.dim() == 3:
        h = h[:, -1, :]
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
            - If h is None: h is extracted from a full forward and stays
              connected to model params (full gradient flow, as before).
            - If h is given: h is used AS-IS instead of the forward (the
              forward is skipped). A ready leaf h (requires_grad=True,
              grad_fn=None) is used IN PLACE — its .grad receives the
              backpropagated derivative d J_norm/d h. J_norm still depends
              on the decoder WEIGHTS between layer_idx and the final norm
              (the decode pass), so gradients also reach those params; only
              the params of layers <= layer_idx / embeddings are excluded.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
        h: pre-computed hidden state at layer_idx (reuse from a prior forward,
            e.g. KL-forward in exp17). Respected in BOTH create_graph paths
            (v6.0 bug-fix: the create_graph=True path previously ignored it
            and unconditionally re-extracted h from a full forward). Not
            renamed to h_override for backward compatibility; the asymmetry
            with compute_jacobian_for_tokens(h_override=...) is intentional.
    """
    model.eval()
    n_layers = _n_layers(model)
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        if h is None:
            # Full forward to get h at layer_idx — connected to all model params + hooks
            with torch.enable_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        # Normalize [1, D] / [D] -> [1, 1, D]; keep any existing grad graph intact.
        h = h.reshape(1, 1, -1) if h.dim() in (1, 2) else h
        model_dtype = next(model.parameters()).dtype
        assert h is not None  # narrowed: param was Tensor|None
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            if h.requires_grad:
                # Caller passed a ready leaf that already requires grad
                # (e.g. a representation-holonomy h_leaf). Use it IN PLACE:
                # detach() would create a NEW leaf and the caller's tensor
                # would never receive .grad. It is already a leaf (grad_fn
                # None), so no graph surgery is needed.
                pass
            else:
                # Fully frozen graph: no trainable path reaches this layer
                # (e.g. linear_attention pair with no LoRA patch, or a
                # no_grad caller), OR a disconnected h was passed in. Make h
                # a leaf so the decoder pass below still yields a
                # differentiable h_norm; J_norm values are identical to the
                # connected case.
                h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm_compat(
                    model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            if h is None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                assert outputs.hidden_states is not None  # requested
                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
            assert h is not None
            h = h.to(next(model.parameters()).dtype)
        return _last_layer_jacobian(h, _lm_module(model).norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = text_model.config.layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J
def _lm_module(model):
    """The text model with .layers/.norm/.rotary_emb.
    Flat CausalLM: model.model IS the text model. Multimodal wrapper
    (Qwen3_5ForConditionalGeneration): model.model.language_model.
    """
    mm = model.model
    if hasattr(mm, "layers"):
        return mm
    return mm.language_model
def _n_layers(model) -> int:
    """num_hidden_layers for flat or wrapper configs."""
    cfg = model.config
    return int(
        cfg.num_hidden_layers
        if hasattr(cfg, "num_hidden_layers")
        else cfg.text_config.num_hidden_layers
    )
def _layer_types(model) -> list:
    """layer_types for flat or wrapper configs."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
def _run_decoder_to_norm_compat(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """_run_decoder_to_norm over _lm_module (flat or nested model)."""
    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)
def h_norm_at_perturbed_h(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    h: torch.Tensor,
) -> torch.Tensor:
    """h_norm after the prefix, with the l_S hidden state pinned to `h`.
    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,
    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);
    the pinning is done on a copy — the input is not mutated.
    """
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.no_grad():
        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
def _forward_h_norm(
    model,
    layer_idx,
    input_ids,
    attention_mask,
    h: torch.Tensor | None = None,
):
    """Forward once -> (h, h_norm), h graph-connected at layer l_S.
    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):
    the prefix pass runs before the model forward (hook-safe), h is the
    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head
    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)
    (last layer) or decoder-to-norm (otherwise).
    h (optional): reuse a pre-computed l_S hidden state instead of doing a
    fresh full forward (v23.1: exp17's KL forward already produced it, so
    the refresh reuses it and only runs the cheap decoder-tail-to-norm).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.enable_grad():
        if h is None:
            # standalone: full forward to obtain h (original A4 behavior)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        assert h is not None
        h = h.to(next(model.parameters()).dtype)
        if h.grad_fn is None:
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            h_norm = _run_decoder_to_norm_compat(
                model, prefix_cache, h, S, layer_idx
            )
    return h, h_norm
def bridge_objective(cos: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """The bridge objective `Σ_k w_k (1 − cos_k)` — ONE formula for the thing that is
    OPTIMIZED and the thing that is REPORTED (audit4 M1).
    Before this, `bridge_rows_backward` minimized the unweighted mean `(1−cos).mean()`
    while `bridge._weighted_cosine_loss` reported `Σ w(1−cos)` with `w = p_L/Σp_L`, so
    ""gap improved"" was not a statement about the trained objective. Both sides call
    this function now.
    `w` is used AS GIVEN: it must be a probability vector over the rows you pass in, and
    the CALLER normalizes. Deliberate — the chunked path normalizes `p` once and then
    slices, so per-chunk losses add up to the global objective. Normalizing inside here
    would renormalize every chunk by its own weight mass: chunk-size dependent, and it
    quietly changes the objective (this is exactly what the chunk-invariance test caught).
    The objective stays a convex combination like the old mean — same scale, no LR
    retuning — but concentrated: on sharp `p` the top token carries most of the gradient,
    by design. Rows dropped by a validity mask must enter with `w = 0` (the metric masks
    them at its caller; on unmasked rows the two sides are identical).
    """
    if cos.shape != w.shape:
        raise ValueError(f"bridge_objective: cos {tuple(cos.shape)} vs w {tuple(w.shape)}")
    return (w.reshape(-1).to(cos.dtype) * (1.0 - cos)).sum()
def bridge_rows_backward(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    J_tgt_p: torch.Tensor,
    extra_loss: torch.Tensor | None = None,
    chunk_size: int = 8,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> tuple[torch.Tensor, float]:
    """v23 A4: chunked bridge backward (the v22 OOM fix).
    Forward once + J_bar once, then build each chunk of K rows and backprop
    its `bridge_objective` loss immediately (extra_loss — the KL — is
    backproped with the first chunk). Backproping per chunk frees that
    chunk's 2nd-order row-graphs at once, so peak = chunk_size + 1 (the
    shared J_bar) second-order graphs, not K. Per-chunk gradients are
    identical to a single backward on the total weighted loss
    `Σ_k (p_k/Σp)(1 − cos_k)` — the SAME functional `bridge.jacobian_gap`
    reports (audit4 M1: the objective must be the metric; it used to be the
    unweighted mean).
    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP
    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).
    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this
    does NOT redo the full forward — only the cheap decoder-tail-to-norm
    runs. exp17's KL forward already produced the same h.
    """
    import time as _time
    K = len(token_set)
    w_obj = p.reshape(-1) / p.reshape(-1).sum()          # metric weights, Σw = 1
    h, h_norm = _forward_h_norm(
        model, layer_idx, input_ids, attention_mask, h=h
    )
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=True,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk's bwd
    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.
    # retain_graph=True keeps the shared h-graph alive for the chunks while
    # the KL's OWN forward graph (a separate forward in exp17) frees here.
    # We deliberately do NOT add it to chunk 0: in a shared-forward test/
    # wiring that would free the h-graph the chunks still need.
    if extra_loss is not None:
        extra_loss.backward(retain_graph=True)
    chunks: list[torch.Tensor] = []
    total_bridge = 0.0
    for c in range(0, K, chunk_size):
        hi = min(c + chunk_size, K)
        is_last = (c + chunk_size) >= K
        rows_c = torch.stack(
            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]
        )
        tgt_c = J_tgt_p[c:hi].float()
        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)
        chunk_loss = bridge_objective(cos_c, w_obj[c:hi])
        # retain_graph on all-but-last keeps the ONE shared h forward graph
        # alive so the next chunk can still build its rows; the chunk's
        # 2nd-order row-graph is released by refcounting once we del it
        # (forward nodes stay, referenced by h / h_norm / J_bar).
        chunk_loss.backward(retain_graph=not is_last)
        total_bridge += chunk_loss.item()
        chunks.append(rows_c.detach())
        del rows_c, cos_c, chunk_loss, tgt_c
        if verbose:
            print(
                f"        bridge_rows: {hi}/{K} "
                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"
            )
    return torch.cat(chunks), total_bridge
def compute_hidden_jacobian_rows(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    create_graph: bool = False,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.
    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =
    softmax(full_logits[token_set])) and the exp17 dense bridge
    (softmax_jacobian(W[token_set]@J_norm, p_L)):
      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]
      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =
      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).
    h stays graph-connected (via _run_decoder_to_norm_compat, which
    routes through _lm_module so both flat CausalLM and the multimodal
    wrapper work), so a caller that patched the target layer's o_proj
    and/or lm_head gets its HVP/bridge gradients from the downstream
    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive
    at once; create_graph=True retains them for a later loss backward,
    create_graph=False frees each chunk after the gradient (for cheap
    no-grad refreshes / evals).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)
    return _compute_jacobian_rows_matrixfree(
        h_norm,
        h,
        W_rows,
        p,
        token_set,
        create_graph=create_graph,
        chunk_size=chunk_size,
        verbose=verbose,
    )
def _compute_jacobian_rows_matrixfree(
    h_norm: torch.Tensor,
    h: torch.Tensor,
    W_rows: torch.Tensor,
    p: torch.Tensor,
    token_set: list[int],
    create_graph: bool = True,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.
    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),
    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;
    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to
    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes
    instead of a retained [D,D] 2nd-order matrix (v22 OOM).
    Chunked (A4): only chunk_size row-graphs alive at once.
    """
    import time as _time
    K = len(token_set)
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=create_graph,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row
    rows = []
    for c in range(0, K, chunk_size):
        for local, i in enumerate(range(c, min(c + chunk_size, K))):
            row = p[i] * (_row_grad(W_rows[i]) - J_bar)
            rows.append(row)
        if verbose:
            done = min(c + chunk_size, K)
            el = _time.time() - start
            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")
    out = torch.stack(rows)
    elapsed = _time.time() - start
    if verbose:
        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")
    return out


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_gap(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 0.0,
) -> float:
    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.
    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted
    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting
    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy
    `jacobian_bridge` alias (which silently returned the GAP direction despite
    its name) was removed after being identified as an inversion trap — the
    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),
    not here.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)
    corrects for non-orthogonal embeddings, so identity (J_L == J_S,
    E_L == E_S) yields gap ~0.
    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).
    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked
    (no decay filtering). Historical default was 1e-6; the paper's
    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,
    ~1e-4 at L18) can be passed explicitly to enable decay masking.
    Epsilon convention (docs/audit4.pdf M3): paper §2.4 writes the cosine
    denominator as ``‖J‖·‖J_target‖ + ε``; the code uses ``F.cosine_similarity``
    (divide by each norm with ``clamp_min=1e-12``). On live rows the two differ
    by ``≲1e-9`` relative; on near-zero rows the additive form is WORSE — it
    turns the cosine into an arbitrary small number, while clamping keeps the
    direction. The formula-vs-code gap is therefore documented, not closed:
    closing it would move published bridge numbers for no semantic gain.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [K, D_L] through E_L^T -> [K, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [K, V] through E_S @ G_inv -> [K, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True, map_location="cpu")
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Hashes shape/device header + the first row, the last row, and up to 9
    deterministic mid-rows (uniformly spaced, seeded from row count so the
    sample is stable across runs but covers the interior). First/last rows
    alone would collide for matrices that share boundary rows (e.g. PAD/UNK)
    but differ internally. Hashing the full 248k×1024 matrix would be
    prohibitively expensive.
    KNOWN SLOP, kept on purpose (audit4 Q4): rows 0 and V−1 are fed to the
    digest twice (once explicitly, once through ``idx``), ``device`` is part of
    the key so a CPU and a GPU run get separate files, and the torch version is
    NOT in the key (a LAPACK change could alter the pseudoinverse in the last
    bits). None of these can cause a false cache HIT — they cost bytes, not
    correctness — and changing any of them invalidates the whole on-disk
    ``results/.gramians`` cache (a full (EᵀE)⁻¹ recompute for a 248k×1024
    embedding), so the key is frozen until something actually needs it.
    """
    V = E.shape[0]
    idx = [0, V - 1]
    if V > 4:
        step = (V - 1) // 10
        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    for i in sorted(set(idx)):
        data += E[i].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm <= threshold (sentinel rows).
    threshold=0.0 (default): only exactly-zero rows are masked.
    threshold>0: decay masking (absolute threshold; the paper's relative
    tau_l = 0.01*median is applied by the caller, E-6 calibration).
    """
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """The reported bridge metric (LOWER = WORSE alignment): `Σ_k w_k(1−cos_k)`,
    `w = probs/Σprobs` over the valid rows. This is THE SAME functional that
    `jacobian.bridge_objective` optimizes — one formula for objective and metric
    (audit4 M1), so ""gap improved"" means the trained loss improved.
    """
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    return bridge_objective(cos_sim, probs[mask] / probs[mask].sum()).item()


In [ ]:
import torch
def _validate(jacobians: list[torch.Tensor]) -> None:
    if not jacobians:
        raise ValueError("jacobians: empty list")
    ref = jacobians[0]
    for J in jacobians[1:]:
        if J.shape != ref.shape:
            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")
        if J.device != ref.device:
            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample
    (document). Cross-sample pooling is invalid — see module docstring /
    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.
    Args:
        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [min(n_components, K)] sorted descending (K < n_components
        possible in degenerate cases; in practice K = |𝒮| ≥ 32).
        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.
    """
    _validate(jacobians)
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def compute_esg_per_sample(
    sample_jacobians: list[list[torch.Tensor]],
    n_components: int = 32,
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.
    Args:
        sample_jacobians: list over samples; each entry is a list of the
            sample's per-position Jacobians [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: list [n_samples] of [n_components].
        eigenvectors: list [n_samples] of [n_components, K].
    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),
    never on the matrices (E-0.1: cross-sample J's are near-orthogonal).
    """
    if not sample_jacobians:
        raise ValueError("sample_jacobians: empty list")
    eigenvalues = []
    eigenvectors = []
    for Js in sample_jacobians:
        ev, evec = compute_esg(Js, n_components=n_components)
        eigenvalues.append(ev)
        eigenvectors.append(evec)
    return eigenvalues, eigenvectors
def cross_positional_gramian(
    J_t: torch.Tensor,
    J_tp: torch.Tensor,
) -> torch.Tensor:
    """Cross-positional Gramian Δ_U(t, t') = J_t @ J_tp.T.
    Args:
        J_t:  [K, D] gauge-free Jacobian at position t.
        J_tp: [K, D] gauge-free Jacobian at position t'.
    Returns:
        [K, K] Gramian (symmetric iff t == t').
    Note:
        - NOT a pooling operation: each call uses exactly two positions.
          See module docstring for intra-trajectory pooling (compute_esg)
          and cross-sample pooling validity (E-0.1).
        - When t == t', result equals the inner-product part of
          compute_esg([J_t]) for N=1: compute_esg([J_t])[0][0] ==
          eigenvalue_0(Δ_U(t, t)). Checked in selfcheck.
        - Caller must ensure J_t, J_tp come from the same model / edit /
          token_set; the function does not check semantic compatibility,
          only shape / device / dtype (via _validate).
    """
    _validate([J_t, J_tp])
    return J_t.float() @ J_tp.float().T
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
import hashlib
import json
import os
import re
import warnings
from collections import Counter
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
# --- закрепление пула (PLAN «Запрос 11»: все результаты воспроизводимы) -------------------
# Порядок streaming-потока на Hub ничем не закреплён, поэтому «первые подходящие строки»
# меняются от прогона к прогону: E-0.1 при идентичных params дал другой top-token набор
# (pk_hash 5c12c174… -> 30d82ade…) и сдвинул C_b(sigma_d) до x31. Пин = сырые тексты принятых
# строк + отпечаток; при наличии пина поток не читается вообще, значит прогон воспроизводим
# и офлайн. Тексты (не тензоры): input_ids по-прежнему производит токенизатор.
PIN_DIR_ENV = "PRISM_CALIB_PIN_DIR"
PIN_WRITE_ENV = "PRISM_CALIB_PIN_WRITE"      # =1: разрешить записать новый пин
PIN_REQUIRE_ENV = "PRISM_CALIB_REQUIRE_PIN"  # =1: отсутствие пина = громкий отказ
LAST_POOL_FPS: dict[str, str] = {}           # domain -> отпечаток последнего загруженного пула
def pool_fingerprint(texts: list[str]) -> str:
    """Отпечаток пула по сырым текстам (от токенизатора не зависит)."""
    h = hashlib.sha256()
    for t in texts:
        h.update(t.encode("utf-8"))
        h.update(b"\x00")
    return h.hexdigest()[:16]
def _default_pin_dir() -> Path | None:
    """Каталог пинов по умолчанию, или None если искать негде.
    В flattening-Kaggle-ядре модулей нет на диске и `__file__` не определён (упавший exp3:
    `NameError: name '__file__' is not defined` прямо в `_pin_path`) — значит путь репозитория
    строится только когда модуль реально лежит в репозитории; иначе пин ищется по `PIN_DIR_ENV`,
    а без него — поток (или громкий отказ при `PIN_REQUIRE_ENV`).
    """
    try:
        here = Path(__file__).resolve()
    except NameError:
        return None
    return here.parent.parent / "data" / "calib_pins"
def _pin_path(domain, model_name, n_samples, min_length, max_length,
              save_offset_mapping) -> Path | None:
    root = os.environ.get(PIN_DIR_ENV)
    base = Path(root) if root else _default_pin_dir()
    if base is None:
        return None
    slug = re.sub(r"[^0-9A-Za-z.]+", "_", model_name)
    mode = "offset" if save_offset_mapping else "plain"
    return base / f"{domain}__{slug}__n{n_samples}__min{min_length}__max{max_length}__{mode}.json"
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable, with a
    RuntimeWarning: a fallback run silently poisons cross-domain comparisons
    (E-13/E-27 T_domain), so the substitution must be visible in the log.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/
        # gated dataset would otherwise pass the check and fail inside
        # load_calibration_data (loud crash instead of the intended fallback).
        return ds_id, cfg, split, field
    except Exception:
        warnings.warn(
            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "
            f"Domain labels in this run are INVALID for cross-domain "
            f"comparisons (E-13/E-27).",
            RuntimeWarning,
            stacklevel=2,
        )
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 64,
    min_length: int = 64,
    n_samples: int = 4,
    domain: str = "wiki",
    save_offset_mapping: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,
    N=4); experiments pass explicit lengths. domain: 'wiki' (C4), 'math'
    (OpenMathInstruct-2), 'code' (apex-agents), 'chat' (ultrachat). Falls back
    to wiki if the target dataset is unavailable. save_offset_mapping: adds
    `offset_mapping` to each sample (E-31 sentence boundaries).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    pin = _pin_path(domain, model_name, n_samples, min_length, max_length, save_offset_mapping)
    texts: list[str] | None = None
    ds_id = cfg = split = field = None
    if pin is not None and pin.exists():
        meta = json.loads(pin.read_text(encoding="utf-8"))
        texts = list(meta["texts"])
        print(f"[calib] {domain}: пин {pin.name}, строк={len(texts)}, "
              f"pool_fp={meta['fp']}, поток не читается", flush=True)
    elif os.environ.get(PIN_REQUIRE_ENV) == "1":
        raise RuntimeError(
            f"нет пина пула {pin or f'каталог пинов недоступен — задайте {PIN_DIR_ENV}'} при "
            f"{PIN_REQUIRE_ENV}=1: прогон на потоке с Hub недопустим, "
            f"иначе числа не воспроизводимы. Сгенерируйте пин с {PIN_WRITE_ENV}=1 один раз и "
            f"закоммитьте его."
        )
    else:
        ds_id, cfg, split, field = _load_dataset_for_domain(domain)
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    # NOTE (audit4 Q9): the offset-mapping path tokenizes with
    # add_special_tokens=False, the plain path with the tokenizer default (True).
    # Deliberate, not a bug: offset_mapping must stay aligned 1:1 with the text
    # span (special tokens carry no span), and the two paths feed different
    # consumers. Consequence to keep in mind: a token set / calibration built
    # through the offset path is NOT the same sequence as the mainline E-16
    # calibration of the same text — do not mix their token sets.
    accepted: list[str] = []
    # Из пина идут уже извлечённые тексты (извлечение = _extract_text_from_sample на момент
    # записи пина); с потока идут сырые строки.
    rows = ((t, None) for t in texts) if texts is not None else ((None, raw) for raw in dataset)
    for pinned_text, raw in rows:
        text = pinned_text if pinned_text is not None else _extract_text_from_sample(raw, field)
        if not text:
            continue
        if save_offset_mapping:
            # Токенизатор вызывается на ОДНОЙ строке, поэтому input_ids — плоский список id, а
            # offset_mapping — список пар; индексация `[0]` здесь была багом (len(int) падал на
            # первом же вызове, найденном E-45b).
            toks = tokenizer(
                text, max_length=max_length, truncation=True,
                add_special_tokens=False, return_offsets_mapping=True,
            )
            if len(toks["input_ids"]) >= min_length:
                accepted.append(text)
                samples.append({
                    "input_ids": toks["input_ids"],
                    "offset_mapping": torch.tensor(
                        toks["offset_mapping"], dtype=torch.int32
                    ),
                })
        else:
            tokens = tokenizer(
                text, max_length=max_length, truncation=True,
                return_tensors="pt",
            )
            if tokens["input_ids"].shape[1] >= min_length:
                accepted.append(text)
                samples.append(tokens)
        if len(samples) >= n_samples:
            break
    if texts is None:
        texts = accepted
        fp = pool_fingerprint(texts)
        print(f"[calib] {domain}: поток с Hub, строк={len(texts)}, pool_fp={fp}", flush=True)
        if os.environ.get(PIN_WRITE_ENV) == "1":
            if pin is None:
                raise RuntimeError(
                    f"{PIN_WRITE_ENV}=1, но каталога пинов нет: {PIN_DIR_ENV} не задан, а "
                    "`__file__` недоступен (flattening-ядро). Задайте PRISM_CALIB_PIN_DIR "
                    "на путь, который попадает в вывод ядра."
                )
            pin.parent.mkdir(parents=True, exist_ok=True)
            pin.write_text(
                json.dumps(
                    {
                        "protocol": {"domain": domain, "model": model_name, "n_samples": n_samples,
                                     "min_length": min_length, "max_length": max_length,
                                     "offset_mapping": bool(save_offset_mapping)},
                        "source": {"dataset": ds_id, "config": cfg, "split": split, "field": field},
                        "fp": fp,
                        "texts": texts,
                    },
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            print(f"[calib] {domain}: пин записан -> {pin}", flush=True)
    else:
        fp = pool_fingerprint(texts)
        if len(samples) < n_samples:
            raise RuntimeError(
                f"пин {pin.name} дал {len(samples)} строк вместо {n_samples}: токенизатор или "
                f"фильтр длины изменились относительно момента записи пина (pool_fp={fp}). "
                f"Молча читать другой пул нельзя — числа стали бы невоспроизводимыми."
            )
    LAST_POOL_FPS[domain] = fp
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Frequency token set: the K tokens that most often appear in the per-sample
    top-K of the LAST position (`logits[:, -1]`) across the calibration pool.
    One basis for every position, so a stored p-row is SUB-stochastic (~half of the
    exp27b positions carry < 0.5 of the softmax mass) — that basis is what all
    published Jacobian numbers sit on. The positional convention (each position gets
    its own top-K, so p is a real truncation) is produced by
    `experiments/exp27b_dense.py: column_mode="positional"` (cache exp27c), not here;
    "union top-K at fixed K" is not definable, since the union of per-position top-K
    exceeds K (audit4 Q10 verdict round). Never mix numbers from the two bases:
    the basis belongs in the ChannelTag slice.
    """
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float32
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention.
    dtype defaults to float32: the tested protocols run on CPU, where fp16
    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.
    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller's
    setup, not here.
    """
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model.
    The bridge projects through the AN-embedding matrix W (§3/§5). For the
    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint
    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection
    would silently use the wrong matrix. We therefore warn ONCE per model if
    lm_head.weight is not the same storage as embed_tokens.weight — a loud,
    non-fatal signal that the W = E assumption does not hold for this model.
    """
    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]
    _warn_if_not_tied(model, emb)
    return emb
def _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:
    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)
    if lm_head is None:
        return
    if not lm_head.data_ptr() == emb.data_ptr():
        # data_ptr identity covers tied weights; fall back to value equality
        # for the (rare) case of a tied-but-copied view.
        if not torch.equal(lm_head.detach(), emb):
            warnings.warn(
                f"extract_embeddings: lm_head.weight is NOT tied to "
                f"embed_tokens.weight (model={type(model).__name__}). The "
                f"bridge projects through E assuming W = E (§3); with untied "
                f"weights the an-embedding matrix is lm_head.weight and the "
                f"gap/bridge numbers are silently wrong. Extract lm_head "
                f"explicitly if this is intended.",
                UserWarning,
                stacklevel=2,
            )


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(
            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"
        )
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
#!/usr/bin/env python3
"""AT0_8 — T8.1 «Пороговое возбуждение» (multi-observable threshold), v1 (до прогона).
ТЗ: docs/TZ_add_4-8.md §8 (T8-FALS); пре-рег: docs/AT0_8_prereg.md.
Вопрос: пороговое рождение локализованного объекта («солитон») при λ-интерполяции весов
W(λ) = λ·W_trained + (1−λ)·W_random vs генерическое включение компетентности.
Контракт (пре-рег §2–§8):
- сетка LAM13 = 11 точек T8.1 + 2 точки T9.1 (общий прогон; вердикт T8.1 — по LAM11);
- 3 сида [11,12,13]; ВСЕ параметры рандомизированы при λ=0 (вкл. 1-D), sd_r = randn·std;
- наблюдаемые: dip/hump/plateau (curves_series h23, пин e64_gpu.py), nll_main (гейт S4),
  nll_holdout (4 дока math/code/chat), margin (top1−top2), pr_sigma/eff_rank (SVD h23),
  rep16/loopy (greedy-генерация 256 ток., prompt+gen);
- λ_c: u(λ) = (O(λ)−O(0))/(O(1)−O(0)), первое удержанное ≥2 точек пересечение u=0.5;
  монотонность |Spearman(λ,O)|≥0.9 (seed-avg, ДО прогона); overshoot u>1 — флаг;
  бисекция ≤5 ит./≤0.02 (union ≤6 точек); CI по сидам (разброс >1 шага ⇒ not_resolved);
- вердикты V3 (патология) > V2 (λ_c^dip отделён от ≥1 из 4) > V1 (все совпадают) > V0;
- health ray (dip_band_hinge, loopy, nll_holdout) + человекочитаемый sample в каждой (λ,сид).
Гейты (до записи; провал = артефакта нет): S1 config; S2 λ=1 побитово = base (sha тензоров);
S2b λ=0 все тензоры случайны; S3 полнота = точное равенство множеств (не truthiness);
S4 nll в каждой записи; S5 finite + pr_sigma≥1 + eff_rank≥1; S6 disjoint; S7 sha e64_gpu.py;
S8 health ray + sample; S9 бисекция.
Smoke: --smoke (2 λ × 1 сид × 2 дока × 1 holdout + генерация, в /tmp/at08_smoke/) — полный
код-путь + гейты на малом масштабе; λ_c-машина в смоуке не считается (сетка вырождена).
Полный прогон — Kaggle (T4 GPU fp32; CPU ≈ 10 ч — тяжёлый, см. пре-рег §9).
"""
import hashlib
import json
import math
import os
import sys
import time
# CPU-политика серии (AT0_4): 4 тред ДО импорта torch (бенч 51.7 с/forward снят на 4);
# transformers.imports поднимает fla (triton) и крашится "0 active drivers" на CPU —
# чистый torch-fallback = референсный путь серии: ОДНАКОВЫЕ операции в смоуке (CPU)
# и полном прогоне (Kaggle GPU) — численная согласованность между ними.
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "4"
sys.modules["fla"] = False
sys.modules["fla.ops"] = False
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
torch.set_grad_enabled(False)
torch.set_num_threads(4)
# ── закреплённые константы (пре-рег §2–§4) ─────────────────────────────────────
MODEL_LOCAL = "/data/Qwen/Qwen3.5-0.8B"
MODEL_HF = "Qwen/Qwen3.5-0.8B"
try:
    _REPO = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:  # notebook (Kaggle-кернел): __file__ отсутствует
    _REPO = os.getcwd()
# Функции e64 (dcurve/vpath_units/curves_series) — встроены VERBATIM из
# experiments/kaggle_e64gpu/e64_gpu.py (sha целого файла — E64_FILE_SHA256;
# sha встроеного источника трёх функций — E64_FNS_SHA256 — проверяется ДО exec;
# без файлового доступа — кернел self-contained). Скрипт не читает e64_gpu.py.
E64_FILE_SHA256 = "8c517e18f801fa877dfde9265ed5ce18df1e7d3af705194fc01ff6b150cf7633"
E64_FNS_SHA256 = "3c215a72f2f1753e8b873929094889c6cf0487adecb591f7cef583519aa906d4"
E64_FNS_SRC = '''def dcurve(P,kk):
    """сериийный оценщик (exp64_b4_battery/A5): median cos по t, без средних."""
    Pn=P/np.maximum(np.linalg.norm(P,axis=1,keepdims=True),1e-12)
    return np.array([float(np.median(np.einsum("ij,ij->i",Pn[:-k],Pn[k:]))) for k in kk])
def vpath_units(u):
    """сериийный Vpath: ортогонализованная составляющая шага между единичными состояниями."""
    Un=u/np.maximum(np.linalg.norm(u,axis=1,keepdims=True),1e-12)
    Vs=[]
    for t in range(len(Un)-1):
        a=float(Un[t]@Un[t+1]); vv=Un[t+1]-a*Un[t]
        if np.linalg.norm(vv)>1e-6: Vs.append(vv/np.linalg.norm(vv))
    return np.array(Vs)
def curves_series(Vh):
    """Vh [T,H] сырые h -> {dip,hump,plateau,curve{}} в ТОЧНОСТИ батареи B4 (median, ортогональн.шаг)."""
    Vh=Vh.astype(np.float64)
    u=Vh/np.maximum(np.linalg.norm(Vh,axis=1,keepdims=True),1e-12)
    P=vpath_units(u)
    kk=np.arange(1,min(96,max(2,len(P)//3))+1)
    c=dcurve(P,kk)
    return dict(dip=float(c[0]),hump=float(c[1:8].max()) if len(c)>8 else float('nan'),
        plateau=float(np.median(c[8:])) if len(c)>9 else float('nan'),
        curve={str(int(k)):float(x) for k,x in zip(kk,c)})
'''
LAM13 = [0.0, 0.05, 0.1, 0.2, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.85, 1.0]
LAM11 = [0.0, 0.2, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.85, 1.0]
LAM11_IDX = {lam: i for i, lam in enumerate(LAM11)}
SEEDS = [11, 12, 13]
T_GEN = 256
PLEN = 32
EOS_ID = 248046          # tokenizer Qwen/Qwen3.5-0.8B: eos_token_id
L_H = 23                 # слой для h (выход последнего слоя; exp71/E-64 конвенция)
MAIN_SHA256 = "4ea0f28771ab1411d8050ee38f8e873c44489758ed81e9586b519cdddad17fd9"
HOLDOUT_SHA256 = "7ac1199e9689e0a2236488824378039e24d795304948f84b5e725e71290efa95"
POOL_FPS = {"chat": "4d3aab66d0e7aac6", "code": "e02cdb39ed828cb2", "math": "1c01d1447c7018a7", "wiki": "144c722df3046f2d"}
MAIN_DOCS = [
        ("wiki_0", [10812, 4811, 43768, 3124, 34639, 10454, 303, 8637, 282, 4120, 0, 198, 5236, 488, 1318, 310, 615, 2577, 506, 3154, 17387, 43768, 30, 1394, 668, 599, 279, 6429, 11, 2113, 411, 383, 678, 12786, 1381, 13, 7548, 11, 5930, 220, 17, 17, 302, 4973, 4196, 3124, 43768, 31424, 11, 17974, 18831, 347, 494, 436, 33961, 277, 51416, 32085, 13, 1216, 668, 381, 12266, 264, 47308, 2119, 523, 364, 4858, 852, 6587, 310, 615, 2577, 440, 836, 55422, 7129, 13, 198, 1465, 668, 4390, 488, 4156, 488, 1144, 310, 1366, 310, 19643, 303, 264, 20437, 7116, 43768, 10391, 11, 2583, 12178, 11, 18066, 11, 84137, 11, 12668, 6381, 321, 79062, 11, 5346, 74578, 321, 3807, 1928, 13, 198, 760, 2695, 310, 381, 303, 279, 523, 369, 393, 18, 20, 791, 1637, 11, 321, 364, 66956, 424, 369, 1845, 13, 44429, 303, 279, 2695, 668, 381, 2892, 264, 259, 32553, 466, 20404, 263, 321, 488, 668, 381, 44854, 10156, 314, 1754, 12668, 421, 369, 9734, 13]),
        ("wiki_1", [68610, 303, 359, 18955, 9787, 1543, 31042, 318, 16, 15, 13, 22, 20973, 3727, 539, 3731, 726, 72, 23, 22, 11, 4209, 220, 17, 15, 11, 220, 17, 15, 16, 17, 13, 198, 40, 2908, 2597, 264, 220, 20, 15, 15, 9231, 5138, 6338, 321, 264, 220, 17, 19, 15, 9231, 35160, 13, 198, 4350, 4313, 310, 14519, 1608, 12977, 15092, 585, 2688, 2574, 279, 1412, 328, 2536, 3213, 3433, 383, 12977, 29739, 310, 14519, 1, 198, 3850, 353, 12752, 914, 599, 310, 635, 421, 11726, 198, 8347, 6498, 466, 944, 277, 2995, 1518, 21117, 286, 310, 279, 3294, 30, 198, 9947, 20829, 14122, 2356, 25728, 310, 2880, 799, 6338, 310, 279, 975, 13, 353, 2908, 2725, 411, 3679, 2942, 2019, 494, 7875, 66205, 310, 8882, 35160, 321, 353, 25707, 685, 440, 264, 10146, 470, 35160, 6338, 13, 3648, 2923, 488, 599, 310, 5909, 524, 310, 10390, 369, 310, 958, 37582, 24370, 310, 16168, 279, 35160, 430, 29235, 16168, 12490, 462, 8225, 10, 1518, 3604, 279, 14289, 13, 1368, 424, 3571, 7905, 52790, 42331, 11, 1442, 413, 488, 1042, 88323, 635, 279, 14289, 11, 279, 12571, 6338, 2677, 914, 381, 10146, 470, 13, 88323, 5802, 4138, 303, 328, 1152, 3728, 1, 321, 424, 628, 6497, 2880, 264, 7875, 6338, 318, 8761, 579, 9712, 4147, 8, 8366, 264, 8882, 6338, 13, 1368, 488, 3184, 88323, 310, 14289, 264, 6338, 488, 1467, 4045, 10146, 494, 11, 424, 628, 944, 303, 2424, 2880, 3728, 1332, 279, 8833, 6338, 1902, 381, 279, 1788, 1331, 466, 7875, 1056, 279, 6338, 488]),
        ("wiki_2", [4199, 1599, 1142, 15412, 791, 1834, 364, 491, 2660, 30, 198, 68610, 303, 359, 14004, 20685, 24189, 6, 3727, 539, 18042, 53228, 450, 11, 3588, 220, 18, 11, 220, 17, 15, 16, 15, 13, 198, 16, 8, 364, 264, 13238, 3354, 2660, 11, 1092, 579, 279, 1866, 653, 1142, 15412, 791, 1834, 353, 1220, 635, 310, 381, 5902, 30, 198, 17, 8, 1204, 1248, 635, 353, 599, 310, 1042, 821, 2660, 4092, 1518, 353, 628, 1151, 3154, 777, 1447, 15049, 30, 198, 40, 1467, 883, 220, 21, 15, 15, 15, 11509, 20235, 1396, 220, 17, 19, 3979, 364, 220, 16, 15, 2756, 364, 799, 314, 821, 6386, 864, 995, 264, 6529, 491, 7697, 13, 198, 3733, 369, 2250, 1142, 15412, 364, 1396, 314, 1439, 11509, 5353, 748, 38705, 220, 16, 23, 220, 15, 15, 15, 1142, 15412, 1396, 220, 17, 19, 3979, 321, 4161, 6716, 303, 3665, 314, 1602, 44782, 1452, 466, 41339, 290, 13, 1061, 369, 1381, 6970, 220, 18, 3818, 3998, 321, 279, 2660, 369, 22284, 383, 1118, 2081, 364, 264, 2609, 314, 821, 16609, 20209, 13, 198, 5653, 777, 488, 628, 303, 5732, 694, 635, 11048, 19878, 321, 524, 24523, 2329, 913, 3254, 11048, 478, 9468, 310, 279, 1675, 486, 1179, 1238, 220, 16, 2182, 488, 628, 1236, 264, 2330, 20045, 486, 198, 34691, 11, 33897, 11, 488, 5617, 220, 16, 23, 74, 1142, 15412, 264, 1834, 383, 264, 6529, 491, 2660, 30, 2500, 5964, 1467, 488, 6857, 685, 30, 3437, 2993, 314, 10391, 22444, 287, 1467, 1782, 20209, 599, 30]),
        ("wiki_3", [760, 21431, 8486, 314, 11332, 8660, 279, 220, 17, 15, 16, 22, 12, 16, 23, 2812, 1007, 440, 449, 2560, 383, 6859, 421, 2830, 491, 7832, 11, 30575, 11, 8364, 63544, 321, 4131, 6618, 21066, 13, 198, 1596, 513, 11692, 421, 21431, 4007, 668, 381, 279, 58526, 314, 264, 2943, 1007, 11, 393, 20, 22, 17, 3410, 3146, 53905, 17394, 22880, 13, 8439, 279, 20438, 314, 279, 10498, 11, 1004, 7832, 2015, 682, 6237, 310, 9412, 279, 6859, 888, 279, 2943, 4514, 4496, 314, 279, 10498, 13, 198, 91916, 11370, 383, 7348, 11454, 10498, 321, 2463, 10456, 10633, 364, 4007, 303, 21431, 2968, 29394, 11, 37057, 310, 2638, 393, 20, 22, 17, 3410, 303, 10498, 10456, 310, 1870, 321, 7042, 8417, 321, 393, 20, 21, 13, 21, 3410, 303, 10042, 10867, 310, 1761, 16134, 26298, 11, 1680, 430, 3988, 50235, 13, 198, 91916, 11370, 1910, 9542, 310, 10498, 321, 2463, 73460, 10456, 1761, 364, 56508, 4007, 321, 8417, 13, 8916, 310, 3827, 777, 883, 279, 3447, 314, 279, 24518, 64025, 10498, 6420, 13, 198, 91916, 11370, 383, 4470, 13, 220, 23, 11454, 10498, 321, 2463, 10456, 10633, 364, 56508, 4007, 321, 8417, 13, 14502, 777, 883, 1092, 725, 5064, 303, 279, 2463, 73460, 6420, 13]),
        ("wiki_4", [33, 5052, 939, 5699, 28602, 604, 8810, 326, 4738, 310, 469, 36201, 5681, 604, 8810, 469, 5257, 81873, 11, 10964, 11, 17397, 11, 321, 4434, 3429, 13, 198, 2053, 314, 1381, 11, 220, 16, 26791, 1542, 1881, 494, 417, 5052, 939, 5699, 28602, 604, 8810, 318, 56, 6277, 8, 310, 469, 36201, 5681, 604, 8810, 318, 38, 553, 198, 760, 24830, 5257, 494, 417, 5052, 939, 5699, 28602, 604, 8810, 318, 56, 6277, 8, 310, 469, 36201, 5681, 604, 8810, 318, 38, 8, 369, 783, 6277, 62969, 6844, 27634, 1658, 5052, 32, 13073, 318, 16, 17, 17, 20, 16, 8, 421, 2101, 6819, 506, 220, 17, 18, 25, 19, 15, 321, 28712, 310, 506, 220, 17, 16, 25, 16, 20, 13, 1049, 4829, 12805, 220, 17, 16, 25, 18, 20, 3979, 13]),
        ("wiki_5", [760, 8807, 615, 56690, 321, 279, 7604, 615, 62981, 34078, 30, 198, 2126, 369, 424, 279, 8807, 1683, 2086, 321, 1432, 539, 264, 2086, 716, 314, 5426, 30, 198, 5236, 279, 8807, 1831, 11415, 321, 1852, 30, 198, 81203, 1208, 4267, 359, 24929, 8311, 310, 728, 3058, 7851, 1208, 513, 10839, 310, 381, 8807, 13, 198, 81203, 1208, 68789, 310, 381, 8807, 13, 7851, 1208, 1683, 2330, 13, 198, 81203, 1208, 1683, 2526, 13, 7851, 1208, 5077, 383, 10175, 13, 198, 81203, 1208, 5077, 383, 29998, 13, 7851, 1208, 513, 9539, 310, 11587, 5402, 321, 836, 869, 13, 198, 81203, 1208, 1683, 45615, 883, 10910, 321, 19633, 13, 198, 81203, 1208, 513, 7629, 310, 491, 6498, 486, 198, 5236, 1394, 1683, 8807, 466, 7604, 30])
]
HOLDOUT_DOCS = [
("math_0", [32, 6378, 369, 8842, 264, 31687, 8151, 440, 1024, 4627, 13, 2838, 6587, 310, 1236, 2617, 781, 599, 3213, 15271, 7674, 15436, 364, 40897, 13, 2532, 668, 381, 4097, 1208, 2702, 25, 88497, 11, 1024, 1330, 4627, 11, 321, 1024, 6357, 13, 2302, 668, 8070, 220, 18, 2756, 321, 220, 17, 20803, 506, 279, 3013, 9233, 11, 321, 781, 3019, 310, 599, 220, 17]),
    ("math_1", [32, 35197, 682, 220, 16, 15, 15, 19434, 314, 65152, 321, 6587, 310, 3107, 1416, 264, 49693, 13159, 440, 449, 2982, 314, 220, 20, 15, 15, 9057, 19434, 13, 1368, 279, 35197, 1048, 6587, 310, 5106, 264, 220, 20, 70040, 6669, 1752, 2094, 279, 4307, 13159, 11, 1204, 1599, 19434, 314, 65152, 668, 381, 4221, 310, 3107, 1416, 279, 13159, 321, 279, 1752, 30]),
    ("code_0", [51, 1646, 321, 1141, 18254, 15234, 279, 1118, 9665, 314, 279, 10507, 28835, 21800, 2128, 318, 7048, 220, 16, 8, 220, 430, 314, 5632, 220, 16, 11, 220, 17, 15, 17, 20, 13, 4642, 3230, 7002, 4945, 42443, 383, 5589, 220, 17, 15, 11, 220, 17, 15, 17, 20, 13, 561, 18254, 579, 6891, 7009, 2445, 494, 10770, 13, 561, 18254, 20653, 345, 1646]),
    ("chat_0", [9205, 10895, 3668, 310, 3644, 5792, 20730, 318, 76102, 220, 21, 13, 15, 43181, 10082, 2141, 220, 19, 13, 15, 43181, 4130, 42700, 220, 18, 13, 15, 10, 45600, 220, 17, 13, 15, 43181, 33749, 23324, 220, 20, 13, 15, 10, 553, 3437, 6697, 2243, 1044, 353, 1608, 30, 198, 1860, 678, 24521, 6602, 594, 48532, 24521, 13753, 11, 488, 628, 6497, 1420, 279])
]
PROMPT = [10812, 4811, 43768, 3124, 34639, 10454, 303, 8637, 282, 4120, 0, 198, 5236, 488, 1318, 310, 615, 2577, 506, 3154, 17387, 43768, 30, 1394, 668, 599, 279, 6429, 11, 2113, 411, 383]
OBS_ALL = ["dip", "hump", "plateau", "nll_holdout", "rep16", "loopy", "pr_sigma", "eff_rank", "margin"]
FAMILIES = {"field": ["dip", "hump", "plateau"], "leaf": ["pr_sigma", "eff_rank"], "mode": ["rep16", "loopy"]}
DISC_OTHERS = ["nll_holdout", "eff_rank", "loopy", "margin"]
BIS_ORDER = ["dip", "nll_holdout", "eff_rank", "loopy", "margin", "hump", "plateau", "pr_sigma", "rep16"]
BIS_CAP = 12             # union-кап точек бисекции (пре-рег §6.6: раунд 1 = 9 mid, раунд 2 = ≤3)
BIS_WIDTH = 0.02
BIS_ITERS = 5
SP_CRIT = 0.9
BASE_SHAS = {}           # заполняется после загрузки модели (S2)
def _script_sha():
    """sha256 скрипта; в notebook (Kaggle) __file__ отсутствует → None (идентичность
    закреплена коммитом-билдом и E64_FNS_SHA256)."""
    try:
        p = os.path.abspath(__file__)
        if os.path.exists(p):
            return sha256_file(p)
    except NameError:
        pass
    return None
def logline(s=""):
    print(s, flush=True)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()
def check_e64_pin():
    """S7: sha-пин встроенного e64-источника ДО exec (пре-рег S7)."""
    sha = hashlib.sha256(E64_FNS_SRC.encode()).hexdigest()
    if sha != E64_FNS_SHA256:
        raise RuntimeError(f"S7 FAIL: e64-функции sha {sha} != pinned {E64_FNS_SHA256}")
    ns = {"np": np}
    exec(E64_FNS_SRC, ns)
    return ns["curves_series"], sha
def docs_sha_check():
    """S6: встроенные токены = закреплённые sha + disjoint (пре-рег §4/S6)."""
    def sha_ids(pairs):
        h = hashlib.sha256()
        for n, ids in pairs:
            h.update(n.encode())
            h.update(b"\x00")
            h.update(np.array(ids, dtype=np.int64).tobytes())
            h.update(b"\x01")
        return h.hexdigest()
    ms = sha_ids(MAIN_DOCS)
    hs = sha_ids(HOLDOUT_DOCS)
    if ms != MAIN_SHA256:
        raise RuntimeError(f"S6 FAIL: main_sha {ms} != {MAIN_SHA256}")
    if hs != HOLDOUT_SHA256:
        raise RuntimeError(f"S6 FAIL: holdout_sha {hs} != {HOLDOUT_SHA256}")
    mn = {n for n, _ in MAIN_DOCS}
    for n, ids in HOLDOUT_DOCS:
        if n in mn:
            raise RuntimeError(f"S6 FAIL: имя holdout {n} в main")
        for m, mid in MAIN_DOCS:
            if ids[:16] == mid[:16]:
                raise RuntimeError(f"S6 FAIL: 16-ток. префикс {n} == {m}")
    if len(PROMPT) != PLEN:
        raise RuntimeError(f"S6 FAIL: prompt len {len(PROMPT)} != {PLEN}")
    if PROMPT != MAIN_DOCS[0][1][:PLEN]:
        raise RuntimeError("S6 FAIL: prompt != main[0][:32]")
    return ms, hs
def load_model():
    path = MODEL_LOCAL if os.path.isdir(MODEL_LOCAL) else MODEL_HF
    m = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32).eval()
    # Kaggle T4: GPU (fp32, torch-fallback без fla — те же операции, что в CPU-смоуке);
    # локально: cuda нет — остаётся CPU.
    if torch.cuda.is_available():
        m = m.cuda()
    return m, path
def model_facts(m):
    tc = getattr(m.config, "text_config", None)
    cfg = tc if tc is not None else m.config
    lt = getattr(cfg, "layer_types", None)
    if lt is None:
        lt = [type(l).__name__ for l in m.model.layers]
    return {
        "num_hidden_layers": int(cfg.num_hidden_layers),
        "hidden_size": int(cfg.hidden_size),
        "vocab_size": int(cfg.vocab_size),
        "tie_word_embeddings": bool(cfg.tie_word_embeddings),
        "layer_types": [str(x) for x in lt],
        "architecture": m.__class__.__name__,
        "n_tensors": len(m.state_dict()),
        "n_1d_tensors": sum(1 for v in m.state_dict().values() if v.dim() == 1),
    }
def state_shas(m):
    return {k: hashlib.sha256(v.detach().cpu().float().numpy().tobytes()).hexdigest()
            for k, v in m.state_dict().items()}
def make_sd_r(sd0, seed):
    """ВЕСЬ state_dict случайный (вкл. 1-D): randn·std, сид-детерминированно (пре-рег §2).
    Per-ключ генератор с РАЗНЫМИ сидами (seed·1000003 + idx в sorted-порядке): те же
    значения независимо от порядка итерации И независимые значения для тензоров
    одинаковой формы (один сид на все ключи дал бы идентичные randn'ы на shape-дублях)."""
    keys = sorted(sd0)
    out = {}
    for i, k in enumerate(keys):
        v = sd0[k]
        g = torch.Generator(device=v.device).manual_seed(seed * 1000003 + i)
        out[k] = torch.randn(v.shape, generator=g, device=v.device) * v.std()
    return out
def check_randomized(sd_r, sd0):
    """S2b: все тензоры случайны. Крупные (n≥1024): max|diff|>0, |mean|<5σ/√n
    (union-bound по ~200 тензоров), std-отношение ∈[0.8,1.25]. Мелкие (n<1024,
    dt_bias/A_log): MC-шум mean/std велик — max|diff|>0 + std-отношение ∈[0.5,2.0]
    (пре-рег S2b: «всё рандомизировано» — строгое неравенство + масштабовая
    валидность); независимость shape-дублей — отдельно в main."""
    bad = []
    n_1d = 0
    n_small = 0
    for k in sorted(sd0):
        a, b = sd0[k].float(), sd_r[k].float()
        if a.dim() == 1:
            n_1d += 1
        if float((a - b).abs().max()) == 0.0:
            bad.append(k)
            continue
        n = a.numel()
        if n >= 1024:
            # 5σ/√n (не 3σ): ~200 больших тензоров → union-bound; цель — ловить
            # не-случайные (константные/сдвинутые) тензоры, а не гауссовость
            mean_ok = abs(float(b.mean())) < 5.0 * float(b.std()) / math.sqrt(n)
            r = float(b.std()) / max(float(a.std()), 1e-12)
            ratio_ok = 0.8 <= r <= 1.25
        else:
            n_small += 1
            # mean-проверка на мелких (n<1024) статистически неосмысленна
            # (MC-шум mean ~ 1/√n: n=16 → 1.5σ-порог ложно срабатывает часто);
            # случайность гарантируют max|diff|>0 + std-отношение + S2b-независимость
            mean_ok = True
            r = float(b.std()) / max(float(a.std()), 1e-12)
            ratio_ok = 0.5 <= r <= 2.0
        if not mean_ok:
            bad.append(k + ":mean")
        if not ratio_ok:
            bad.append(k + f":stdratio={r:.3f}")
    return bad, n_1d, n_small
def set_interp(m, sd0, sd_r, lam):
    """W(λ) = λ·sd0 + (1−λ)·sd_r. λ∈{0,1} — чистый copy_ (побитово, без арифметики
    1.0·x/0.0·x — защита от −0.0 flip, пре-рег S2)."""
    with torch.no_grad():
        dev = next(m.parameters()).device
        for k, v in m.state_dict().items():
            if lam == 1.0:
                v.copy_(sd0[k].to(dev))
            elif lam == 0.0:
                v.copy_(sd_r[k].to(dev))
            else:
                v.copy_((lam * sd0[k] + (1.0 - lam) * sd_r[k]).to(dev))
class Hook:
    def __init__(self, layer):
        self.val = None
        self.h = layer.register_forward_hook(self)
    def __call__(self, mod, inp, out):
        self.val = out[0] if isinstance(out, tuple) else out
    def remove(self):
        self.h.remove()
def fwd_doc(m, ids, hook):
    """Один forward дока: nll (CE, t≥1), h23 [T,D], margin-медиана (top1−top2, t≥1)."""
    t = torch.tensor([ids], dtype=torch.long, device=m.device)
    with torch.no_grad():
        o = m(input_ids=t, use_cache=False)
    logits = o.logits[0].float()
    nll = float(F.cross_entropy(logits[:-1], t[0, 1:]))
    lg = logits[1:].cpu().numpy()
    srt = np.sort(lg, axis=1)
    margin = float(np.median(srt[:, -1] - srt[:, -2]))
    h = hook.val[0, :len(ids)].detach().float().cpu().numpy()
    return nll, h, margin
def doc_metrics(h, curves_series):
    cur = curves_series(h)
    hh = h.astype(np.float64)
    s = np.linalg.svd(hh, compute_uv=False)
    s2 = s * s
    pr_sigma = float((s2.sum()) ** 2 / max(float((s2 * s2).sum()), 1e-300))
    p = s2 / s2.sum()
    eff_rank = float(np.exp(-float((p * np.log(p)).sum())))
    return {
        "dip": float(cur["dip"]), "hump": float(cur["hump"]), "plateau": float(cur["plateau"]),
        "pr_sigma": pr_sigma, "eff_rank": eff_rank,
    }
def gen_metrics(all_toks, h_all, curves_series):
    """rep16 (16-окна без перекрытия, <12 distinct) + loopy (max ACF k≥2 > 0.95)."""
    T = len(all_toks)
    nwin = max(0, (T - 16) // 16)
    rep16 = None
    if nwin > 0:
        hits = sum(1 for i in range(0, T - 16, 16) if len(set(all_toks[i:i + 16])) < 12)
        rep16 = float(hits) / nwin
    loopy = None
    if T >= 28:
        c = curves_series(h_all)["curve"]
        ks = [k for k in c if int(k) >= 2]
        if ks:
            loopy = int(max(c[k] for k in ks) > 0.95)
    deg = (T - PLEN) < 32
    return rep16, loopy, deg
def generate(m, prompt, hook):
    """Greedy до T_GEN или EOS; h23 по всем позициям (prompt + gen)."""
    x = torch.tensor([prompt], dtype=torch.long, device=m.device)
    cache = DynamicCache(config=m.config)
    with torch.no_grad():
        o = m(input_ids=x, past_key_values=cache, use_cache=True)
    hs = [hook.val[0, -1].detach().float().cpu().numpy()]
    toks = list(prompt)
    for _ in range(T_GEN):
        lg = o.logits[0, -1].float()
        nxt = int(lg.argmax())
        if nxt == EOS_ID:
            break
        toks.append(nxt)
        with torch.no_grad():
            o = m(input_ids=torch.tensor([[nxt]], dtype=torch.long, device=m.device),
                  past_key_values=cache, use_cache=True)
        hs.append(hook.val[0, -1].detach().float().cpu().numpy())
    h_all = np.stack(hs, axis=0)
    return toks, h_all
def run_block(m, sd0, sd_r, lam, doc_names, hold_names, tk_decode, curves_series):
    """Один (λ, сид): форварды main/holdout + генерация. Возвращает cell."""
    set_interp(m, sd0, sd_r, lam)
    hook = Hook(m.model.layers[L_H])
    cell = {"lam": lam, "docs": {}, "holdout": {}}
    try:
        for n, ids in MAIN_DOCS:
            if n not in doc_names:
                continue
            nll, h, margin = fwd_doc(m, ids, hook)
            d = doc_metrics(h, curves_series)
            d["nll"] = nll
            d["margin"] = margin
            d["T"] = len(ids)
            cell["docs"][n] = d
        for n, ids in HOLDOUT_DOCS:
            if n not in hold_names:
                continue
            nll, h, _ = fwd_doc(m, ids, hook)
            cell["holdout"][n] = {"nll": nll, "T": len(ids)}
        toks, h_all = generate(m, PROMPT, hook)
        rep16, loopy, deg = gen_metrics(toks, h_all, curves_series)
        cell["gen"] = {
            "tokens": toks, "T_prompt": PLEN, "T_act": len(toks) - PLEN,
            "rep16": rep16, "loopy": loopy, "degenerate_short": bool(deg),
            "text": tk_decode(toks),
        }
        if lam == 1.0:
            cell["S2_sha_ok"] = (state_shas(m) == BASE_SHAS)
    finally:
        hook.remove()
    return cell
def rankdata(x):
    """Авг. ранги (Spearman с дублями)."""
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return ranks
def spearman(a, b):
    ra, rb = rankdata(a), rankdata(b)
    if ra.std() == 0 or rb.std() == 0:
        return None
    return float(np.corrcoef(ra, rb)[0, 1])
def u_curve(O, grid):
    """u(λ) = (O(λ)−O(0))/(O(1)−O(0)) по grid; None если no_range (пре-рег §6.1)."""
    o0, o1 = O[grid[0]], O[grid[-1]]
    if abs(o1 - o0) < 1e-6 * max(1.0, abs(o0), abs(o1)):
        return None
    return {lam: (O[lam] - o0) / (o1 - o0) for lam in grid}
def crossing_of(u, grid):
    """Первый j: u(λ_j)≥0.5 И u(λ_{j+1})≥0.5 (персистентность) → (λ_j, idx) или None."""
    for i in range(len(grid) - 1):
        if u[grid[i]] >= 0.5 and u[grid[i + 1]] >= 0.5:
            return grid[i], i
    return None
def lam_c_machine(curves, grid, idx_of, obs_list=None):
    """curves: {obs: {seed: {lam: val}}}. Возвращает (per-obs λ_c-блок, seedavg, u_seedavg,
    bisection-интервалы). Бисекция-уточнение — отдельно (нужны доп. форварды)."""
    out = {}
    obs_list = obs_list or OBS_ALL
    seedavg = {o: {lam: float(np.mean([curves[o][s][lam] for s in SEEDS])) for lam in grid}
               for o in obs_list}
    u_sa = {o: u_curve(seedavg[o], grid) for o in seedavg}
    for o in obs_list:
        entry = {"seeds": {}, "ci": None, "refined": None, "outcome": None,
                 "spearman_seedavg": spearman(np.array(grid), np.array(list(seedavg[o].values()))),
                 "max_u": None, "overshoot": False, "u_exit": None,
                 "u_seedavg": {f"{k:.3f}": v for k, v in u_sa[o].items()} if u_sa[o] else None,
                 "bisect_interval": None}
        if u_sa[o] is None:
            entry["outcome"] = "no_range"
            out[o] = entry
            continue
        rho = entry["spearman_seedavg"]
        if rho is None or abs(rho) < SP_CRIT:
            entry["outcome"] = "non_monotone_excluded"
            out[o] = entry
            continue
        entry["max_u"] = max(u_sa[o].values())
        entry["overshoot"] = bool(entry["max_u"] > 1.0)
        sig = float(np.std([curves[o][s][grid[0]] for s in SEEDS]))
        for lam in grid:
            if abs(seedavg[o][lam] - seedavg[o][grid[0]]) > 2.0 * sig:
                entry["u_exit"] = lam
                break
        lcs = []
        for s in SEEDS:
            us = u_curve(curves[o][s], grid)
            if us is None:
                entry["seeds"][s] = None
                continue
            cr = crossing_of(us, grid)
            entry["seeds"][s] = cr[0] if cr else None
            if cr:
                lcs.append(cr[1])
        if not lcs:
            entry["outcome"] = "no_transition"
            out[o] = entry
            continue
        vals = [grid[i] for i in lcs]
        lo, hi = min(vals), max(vals)
        span = idx_of[hi] - idx_of[lo]
        entry["ci"] = [lo, hi]
        entry["outcome"] = "not_resolved" if span > 1 else "resolved"
        # интервал для бисекции: j0 — первое u_seedavg ≥ 0.5 (всегда есть: u(1)=1)
        j0 = next(i for i, lam in enumerate(grid) if u_sa[o][lam] >= 0.5)
        a, b = (0.0 if j0 == 0 else grid[j0 - 1], grid[j0])
        entry["bisect_interval"] = [a, b]
        out[o] = entry
    return out, seedavg, u_sa
def bisection_plan(bis_intervals, computed_lams, order=None):
    """Union-план точек бисекции (пре-рег §6.6): раунд 1 — mid всех интервалов (order),
    раунд 2 — mid оставшихся (если ширина > BIS_WIDTH). Кап BIS_CAP. Возвращает список
    раундов [[(obs, mid), ...], ...] без точек, уже посчитанных."""
    order = order or BIS_ORDER
    rounds = []
    used = set()
    intervals = {o: list(iv) for o, iv in bis_intervals.items() if iv}
    for rnd in range(BIS_ITERS):
        pts = []
        for o in order:
            if o not in intervals:
                continue
            a, b = intervals[o]
            if b - a <= BIS_WIDTH:
                intervals.pop(o)
                continue
            mid = round((a + b) / 2.0, 6)
            if mid in used or mid in computed_lams:
                continue
            pts.append((o, mid))
        if not pts:
            break
        rounds.append(pts)
        for o, mid in pts:
            used.add(mid)
    flat = [p for r in rounds for p in r]
    capped = len(flat) > BIS_CAP
    if capped:
        keep = set()
        rounds2 = []
        for r in rounds:
            nr = []
            for p in r:
                if len(keep) < BIS_CAP:
                    keep.add(p[1])
                    nr.append(p)
            if nr:
                rounds2.append(nr)
        rounds = rounds2
    return rounds, capped
def refine_interval(a, b, u_at):
    """Бисекция по уже посчитанным u-значениям (seed-avg, пре-рег §6.6).
    Возвращает (refined=(a+b)/2, width=b−a); стоп: width ≤ BIS_WIDTH, ≤BIS_ITERS
    итераций, или mid не посчитан."""
    for _ in range(BIS_ITERS):
        if b - a <= BIS_WIDTH:
            break
        mid = round((a + b) / 2.0, 6)
        if mid not in u_at:
            break
        if u_at[mid] >= 0.5:
            b = mid
        else:
            a = mid
    return (a + b) / 2.0, (b - a)
def refine_bisect(obs, iv, cells, seedavg_pre, doc_names, hold_names, lam_list):
    """Бисекция-уточнение одной наблюдаемой (пре-рег §6.6): нормированные u(λ) на
    бисекционных точках + точках lam_list (seed-avg), refine_interval.
    Возвращает (refined, width); (None, None) если no_range (scale ~ 0)."""
    o0, o1 = seedavg_pre[obs][lam_list[0]], seedavg_pre[obs][lam_list[-1]]
    scale = o1 - o0
    if abs(scale) < 1e-12:
        return None, None
    u_at = {}
    for key, c in cells.items():
        if "bis" not in key:
            continue
        lam = c["lam"]
        if obs not in u_at:
            u_at[lam] = {}
        u_at[lam][c["seed"]] = aggregate_cell(c, doc_names, hold_names)[obs]
    u_sa_at = {}
    for lam, vals in u_at.items():
        u_sa_at[lam] = (float(np.mean(list(vals.values()))) - o0) / scale
    # расширяем точками lam_list (также нормированные) для непрерывности
    u_sa_at.update({lam: (val - o0) / scale for lam, val in seedavg_pre[obs].items()})
    return refine_interval(iv[0], iv[1], u_sa_at)
def selftest_lam_c():
    """Детерминированный self-test машины λ_c + verdict-веток (AGENTS: exec точного фрагмента)."""
    A = {0.0: 0.0, 0.2: 0.1, 0.4: 0.3, 0.5: 0.45, 0.55: 0.6, 0.6: 0.7,
         0.65: 0.8, 0.7: 0.85, 0.75: 0.9, 0.85: 0.97, 1.0: 1.0}
    B = [0.0, 0.9, 0.2, 0.95, 0.1, 0.9, 0.3, 0.85, 0.2, 0.8, 1.0]  # немонотонна, O(0)≠O(1)
    B = dict(zip(LAM11, B))
    C = {lam: 0.001 for lam in LAM11}
    curves = {"A": {s: A for s in SEEDS}, "B": {s: B for s in SEEDS},
              "C": {s: C for s in SEEDS}}
    # временная замена OBS_ALL — self-test на 3 псевдо-наблюдаемых
    out, seedavg, us = lam_c_machine(curves, LAM11, LAM11_IDX, ["A", "B", "C"])
    assert out["A"]["seeds"] == {11: 0.55, 12: 0.55, 13: 0.55}, out["A"]["seeds"]
    assert out["A"]["outcome"] == "resolved" and out["A"]["ci"] == [0.55, 0.55]
    assert out["A"]["overshoot"] is False
    assert abs(out["A"]["spearman_seedavg"] - 1.0) < 1e-9
    assert out["A"]["bisect_interval"] == [0.5, 0.55]
    assert out["B"]["outcome"] == "non_monotone_excluded"
    assert out["C"]["outcome"] == "no_range"
    D = dict(A); D[0.55] = 0.6; D[0.6] = 0.4   # выброс 0.55 не удержан (0.6 < 0.5) → первое
                                                # удержанное пересечение — пара (0.65, 0.7)
    out2, _, _ = lam_c_machine({"D": {s: D for s in SEEDS}}, LAM11, LAM11_IDX, ["D"])
    assert out2["D"]["seeds"] == {11: 0.65, 12: 0.65, 13: 0.65}, out2["D"]["seeds"]
    E = dict(A); E[0.7] = 1.2                    # overshoot
    out3, _, _ = lam_c_machine({"E": {s: E for s in SEEDS}}, LAM11, LAM11_IDX, ["E"])
    assert out3["E"]["overshoot"] is True
    # bisection_plan: интервалы [0.5,0.55] и [0.2,0.4] → mid 0.525, 0.3
    # (mid на точке сетки — не планируется: u уже посчитан в main-сетке)
    r, capped = bisection_plan({"A": [0.5, 0.55], "B": [0.2, 0.4]}, LAM11, ["A", "B"])
    mids = sorted(p[1] for rr in r for p in rr)
    assert mids == [0.3, 0.525] and not capped, (mids, capped)
    # bisection_plan cap: 13 интервалов (все шириной > 0.02) → 13 mid > BIS_CAP=12
    # → capped, 12 точек, отсечка по порядку (BIS_ORDER)
    many = {f"O{i}": [0.0, 0.05 + 0.1 * i] for i in range(13)}
    r2, capped2 = bisection_plan(many, LAM11, [f"O{i}" for i in range(13)])
    mids2 = [p[1] for rr in r2 for p in rr]
    assert capped2 and len(mids2) == 12 and len(set(mids2)) == 12, (capped2, mids2)
    # refine: НОРМИРОВАННЫЕ u (контракт функции) — сырые O-значения не сравнивать с 0.5
    # (регрессия BLOCKER: бисекция-рефаин на сырых O давала границу в неверном направлении)
    raw = {0.0: 12.0, 0.5: 8.0, 0.55: 6.0, 1.0: 2.2}   # nll-подобная убывающая кривая
    o0, o1 = raw[0.0], raw[1.0]
    un = {lam: (v - o0) / (o1 - o0) for lam, v in raw.items()}
    assert un[0.55] > 0.5 > un[0.5]
    u525 = (10.0 - o0) / (o1 - o0)   # u(0.525) для сырого 10.0 = 0.204 < 0.5 → a=0.525
    ref_n, w_n = refine_interval(0.5, 0.55, {0.525: u525})
    # сырое 10.0 ≥ 0.5 → refine даст b=0.525 (неверное направление)
    ref_bad, _ = refine_interval(0.5, 0.55, {0.525: 10.0})
    assert ref_n != ref_bad, (ref_n, ref_bad)
    assert abs(ref_n - 0.5375) < 1e-9 and abs(w_n - 0.025) < 1e-9, (ref_n, w_n)
    # refine: две точки → данные кончились на ширине 0.025
    ref2, w2 = refine_interval(0.4, 0.5, {0.45: 0.6, 0.425: 0.3})
    assert abs(ref2 - 0.4375) < 1e-9 and abs(w2 - 0.025) < 1e-9, (ref2, w2)
    # refine: три точки → сходится до ≤0.02 (0.1→0.05→0.025→0.0125)
    ref3, w3 = refine_interval(0.4, 0.5, {0.45: 0.6, 0.425: 0.3, 0.4375: 0.55})
    assert w3 <= 0.02 and abs(ref3 - 0.43125) < 1e-9, (ref3, w3)
    # bisection-рефаин: exec ТОЙ функции, что гонится в main_AT08threshold(Kaggle v1 упал здесь:
    # np.mean(dict_values) — TypeError; теперь list() + общий selftest)
    def _cell(seed, lam, val):
        return {"lam": lam, "seed": seed,
                "docs": {"d1": {"dip": val, "hump": val, "plateau": val, "margin": val,
                               "pr_sigma": 1.0, "eff_rank": 1.0, "nll": val}},
                "holdout": {"h1": {"nll": val}},
                "gen": {"rep16": 0.0, "loopy": 0}}
    cells_syn = {}
    for s in (11, 12, 13):
        cells_syn[f"s{s}_lam0.00"] = _cell(s, 0.0, 12.0)
        cells_syn[f"s{s}_lam1.00"] = _cell(s, 1.0, 2.2)
        cells_syn[f"s{s}_bis0.525"] = _cell(s, 0.525, 10.0)
    sp = {"dip": {0.0: 12.0, 1.0: 2.2}}
    ref_b, w_b = refine_bisect("dip", [0.5, 0.55], cells_syn, sp, ["d1"], ["h1"], [0.0, 1.0])
    # u(0.525) = (10−12)/(2.2−12) = 0.204 < 0.5 → a=0.525 → ref=0.5375, width=0.025
    assert abs(ref_b - 0.5375) < 1e-9 and abs(w_b - 0.025) < 1e-9, (ref_b, w_b)
    return True
def pair_compare(dip_e, o_e):
    """«Совпадает/отделён» по пре-рег §6: per-seed |Δ| в шагах сетки + CI-overlap."""
    if dip_e.get("ci") is None or o_e.get("ci") is None:
        return None
    a_lo, a_hi = dip_e["ci"]
    b_lo, b_hi = o_e["ci"]
    overlap = not (a_hi < b_lo or b_hi < a_lo)
    ns = 0
    for s in SEEDS:
        dv, ov = dip_e["seeds"].get(s), o_e["seeds"].get(s)
        if dv is None or ov is None:
            continue
        ns += int(abs(LAM11_IDX[ov] - LAM11_IDX[dv]) >= 2)
    p_raw = (1.0 / 8.0) * sum(math.comb(3, k) for k in range(ns, 4))  # P(Bin(3,½) ≥ ns)
    p_bonf = min(1.0, p_raw / len(DISC_OTHERS))
    # «совпадает»: per-seed |Δ| ≤ 1 шаг (все сида с λ_c) И CI пересекаются
    all_le1 = True
    for s in SEEDS:
        dv, ov = dip_e["seeds"].get(s), o_e["seeds"].get(s)
        if dv is not None and ov is not None and abs(LAM11_IDX[ov] - LAM11_IDX[dv]) > 1:
            all_le1 = False
    coincide = all_le1 and overlap
    separated = (ns == len(SEEDS)) and (not overlap)
    return {"ci_overlap": overlap, "seeds_ge2steps": ns, "p_raw": p_raw, "p_bonf": p_bonf,
            "coincide": bool(coincide), "separated": bool(separated)}
def verdict(lambda_c, seedavg, us):
    """V3 > V2 > V1 > V0 (пре-рег §7)."""
    dip = lambda_c.get("dip", {})
    dip_outcome = dip.get("outcome")
    # V3: патология (overshoot dip на 0.7 + rep16 выше λ=1)
    u7 = us.get("dip")
    if u7 is not None and "rep16" in seedavg and seedavg["rep16"][0.7] is not None \
            and seedavg["rep16"][1.0] is not None and u7[0.7] > 1.0 \
            and seedavg["rep16"][0.7] > seedavg["rep16"][1.0]:
        return {"outcome": "V3_PATHOLOGY",
                "rule": "u_dip(0.7)>1 и rep16(0.7)>rep16(1) (seed-avg) — патология порога",
                "details": {"u_dip_0.7": u7[0.7], "rep16_0.7": seedavg["rep16"][0.7],
                            "rep16_1.0": seedavg["rep16"][1.0]}}
    if dip_outcome not in ("resolved",):
        return {"outcome": "V0_NOT_RESOLVED",
                "rule": "dip не даёт resolved λ_c",
                "details": {"dip_outcome": dip_outcome,
                            "excluded_non_monotone": [o for o, e in lambda_c.items()
                                                      if e.get("outcome") == "non_monotone_excluded"]}}
    pairs = {}
    sep_any = False
    coin_all = True
    for o in DISC_OTHERS:
        e = lambda_c.get(o, {})
        pc = pair_compare(dip, e)
        if pc is None:
            pairs[o] = {"comparable": False,
                        "reason": f"dip:{dip_outcome} other:{e.get('outcome')}"}
            coin_all = False
            continue
        pc["comparable"] = True
        pairs[o] = pc
        sep_any = sep_any or pc["separated"]
        coin_all = coin_all and pc["coincide"]
    if sep_any:
        return {"outcome": "V2_SOLITON_CANDIDATE",
                "rule": "λ_c^dip отделён от ≥1 из {nll_holdout, eff_rank, loopy, margin} "
                        "(per-seed ≥2 шага + CI не пересекаются)",
                "details": {"pairs": pairs,
                            "note": "порядок на прямой интерполяции весов, НЕ на обучении "
                                    "(TЗ §8.4 п.1) — кандидат идёт в T8.2"}}
    if coin_all:
        return {"outcome": "V1_GENERIC_COMPETENCE",
                "rule": "λ_c^dip совпадает со всеми 4 (per-seed ≤1 шага + CI пересекаются)",
                "details": {"pairs": pairs,
                            "note": "генерическая компетентность; слово «солитон» снимается"}}
    return {"outcome": "V0_NOT_RESOLVED",
            "rule": "смешанный случай (не все совпадения и ни одного отдаления)",
            "details": {"pairs": pairs}}
def families_check(lambda_c):
    """Согласованность внутри семейств (≥2 члена) — пре-рег §6/§8.3."""
    out = {}
    for fam, members in FAMILIES.items():
        pairs = {}
        ok = True
        for i, a in enumerate(members):
            for b in members[i + 1:]:
                ea, eb = lambda_c.get(a, {}), lambda_c.get(b, {})
                if ea.get("ci") is None or eb.get("ci") is None:
                    pairs[f"{a}|{b}"] = {"comparable": False}
                    ok = False
                    continue
                pc = {"ci_overlap": not (ea["ci"][1] < eb["ci"][0] or eb["ci"][1] < ea["ci"][0])}
                all_le1 = True
                for s in SEEDS:
                    va, vb = ea["seeds"].get(s), eb["seeds"].get(s)
                    if va is not None and vb is not None \
                            and abs(LAM11_IDX[vb] - LAM11_IDX[va]) > 1:
                        all_le1 = False
                pc["coincide"] = bool(all_le1 and pc["ci_overlap"])
                pairs[f"{a}|{b}"] = pc
                ok = ok and pc["coincide"]
        out[fam] = {"members": members, "intra_consistent": bool(ok), "pairs": pairs}
    return out
# ── агрегация и health ray ─────────────────────────────────────────────────────
def aggregate_cell(cell, doc_names, hold_names):
    """per-(λ,сид): наблюдаемые = медианы (main: 6 доков, holdout: 4 дока)."""
    ag = {}
    for obs in ("dip", "hump", "plateau", "margin", "pr_sigma", "eff_rank"):
        vals = [cell["docs"][n][obs] for n in doc_names if n in cell["docs"]]
        ag[obs] = float(np.median(vals)) if vals else None
    nlls = [cell["holdout"][n]["nll"] for n in hold_names if n in cell["holdout"]]
    ag["nll_holdout"] = float(np.median(nlls)) if nlls else None
    g = cell.get("gen", {})
    ag["rep16"] = g.get("rep16")
    ag["loopy"] = g.get("loopy")
    return ag
def health_ray(cells_by_seed, doc_names, hold_names):
    """per (сид, λ): (dip_band_hinge, loopy, nll_holdout) — пре-рег S8/§9.6."""
    out = {}
    for s in SEEDS:
        c1 = cells_by_seed.get((s, 1.0))
        dip1 = None
        if c1:
            vals = [c1["docs"][n]["dip"] for n in doc_names if n in c1["docs"]]
            dip1 = float(np.median(vals))
        for key, c in cells_by_seed.items():
            if key[0] != s:
                continue
            lam = key[1]
            if lam == 1.0 or dip1 is None:
                hinge = 0.0 if lam == 1.0 else None
            else:
                vals = [c["docs"][n]["dip"] for n in doc_names if n in c["docs"]]
                if not vals:
                    hinge = None
                else:
                    hinge = abs(float(np.median(vals)) - dip1)
            out[f"s{s}_lam{lam:.2f}"] = {
                "dip_band_hinge": hinge,
                "loopy": c.get("gen", {}).get("loopy"),
                "nll": (float(np.median([c["holdout"][n]["nll"]
                                        for n in hold_names if n in c["holdout"]]))
                        if c.get("holdout") else None),
                "sample": c.get("gen", {}).get("text", "")[:400],
            }
    return out
def canonical_sha(payload):
    st = {k: v for k, v in payload.items() if k != "store_sha256"}
    st["store_sha256"] = None
    return hashlib.sha256(json.dumps(st, sort_keys=True).encode()).hexdigest()
# ── main ───────────────────────────────────────────────────────────────────────
def main_AT08threshold(results_dir: str | None = None):
    t0 = time.time()
    args = sys.argv[1:]
    smoke = "--smoke" in args
    out_dir = "/tmp/at08_smoke" if smoke else (results_dir or os.path.join(_REPO, "results", "AT0"))
    os.makedirs(os.path.join(out_dir, "store"), exist_ok=True)
    # run-log (трекаемый артефакт, пре-рег §11): tee stdout в out_dir/AT0_8_run.log
    _logf = open(os.path.join(out_dir, "AT0_8_run.log"), "w")
    class _Tee:
        def __init__(self, *fs):
            self.fs = fs
        def write(self, s):
            for f in self.fs:
                f.write(s)
        def flush(self):
            for f in self.fs:
                f.flush()
    sys.stdout = _Tee(sys.__stdout__, _logf)
    logline(f"AT0_8 T8.1 {'SMOKE' if smoke else 'FULL'} (py{sys.version.split()[0]}, "
            f"torch{torch.__version__}, device "
            f"{'cuda' if torch.cuda.is_available() else 'cpu'})")
    selftest = selftest_lam_c()
    logline(f"selftest_lam_c: PASS (lambda_c machine + bisection plan + refine)")
    assert selftest
    e64_sha = None
    dcurve = None  # noqa: F841  (в ns-закрытии curves_series; отдельно не используется)
    curves_series, e64_sha = check_e64_pin()
    logline(f"S7: e64_gpu.py sha {e64_sha[:16]}… PASS")
    main_sha, holdout_sha = docs_sha_check()
    logline(f"S6: docs sha {main_sha[:16]}… / {holdout_sha[:16]}… + disjoint PASS")
    m, model_path = load_model()
    facts = model_facts(m)
    assert facts["num_hidden_layers"] == 24 and facts["hidden_size"] == 1024, facts
    logline(f"S1: {facts['architecture']} layers=24 D=1024 vocab={facts['vocab_size']} "
            f"1D-тензоров={facts['n_1d_tensors']} layer_types[0..3]={facts['layer_types'][:4]}")
    BASE_SHAS.update(state_shas(m))
    logline(f"base state: {len(BASE_SHAS)} тензоров, sha зафиксированы (S2-референс)")
    tk = AutoTokenizer.from_pretrained(model_path)
    tk_decode = lambda toks: tk.decode(toks, skip_special_tokens=False)
    sd0 = {k: v.detach().clone() for k, v in m.state_dict().items()}
    if smoke:
        doc_names = [n for n, _ in MAIN_DOCS[:2]]
        hold_names = [n for n, _ in HOLDOUT_DOCS[:1]]
        grid = [0.0, 1.0]
        seeds = [11]
    else:
        doc_names = [n for n, _ in MAIN_DOCS]
        hold_names = [n for n, _ in HOLDOUT_DOCS]
        grid = LAM13
        seeds = SEEDS
    inprog_path = os.path.join(out_dir, "store", "AT0_8_inprogress.json")
    cells = {}
    if os.path.exists(inprog_path) and not smoke:
        try:
            cells = json.load(open(inprog_path))["cells"]
            logline(f"resume: {len(cells)} клеток уже посчитано (скип)")
        except (json.JSONDecodeError, KeyError, TypeError):
            logline("resume: inprogress.json усечён (обрыв) — старт с нуля")
            cells = {}
    sd_r = None
    s2b = {}
    for s in seeds:
        sd_r = make_sd_r(sd0, s)
        bad, n_1d, n_small = check_randomized(sd_r, sd0)
        # S2b-независимость: shape-дубли (например A_log во всех linear-слоях) — РАЗНЫЕ
        # случайные значения (один сид на все ключи дал бы идентичные randn'ы)
        shapes = {}
        for k, v in sd_r.items():
            shapes.setdefault(tuple(v.shape), []).append(k)
        dup_bad = [ks for ks in shapes.values()
                   if len(ks) > 1 and all(torch.equal(sd_r[ks[0]], sd_r[k]) for k in ks[1:])]
        assert not dup_bad, f"S2b FAIL seed {s}: идентичные shape-дубли {dup_bad[:3]!r}"
        s2b[s] = {"bad": bad, "n_1d": n_1d, "n_small": n_small}
        assert not bad, f"S2b FAIL seed {s}: {bad[:5]}"
        logline(f"S2b: seed {s} — все {len(sd0)} тензоров случайны (1-D: {n_1d}) PASS")
        for lam in grid:
            key = f"s{s}_lam{lam:.2f}"
            if key in cells:
                continue
            c = run_block(m, sd0, sd_r, lam, doc_names, hold_names, tk_decode, curves_series)
            c["seed"] = s
            cells[key] = c
            logline(f"  s{s} λ={lam:.2f}: dip={c['docs'][doc_names[0]]['dip']:+.4f} "
                    f"nll_h={c['holdout'][hold_names[0]]['nll']:.3f} "
                    f"T_act={c['gen']['T_act']} rep16={c['gen']['rep16']} "
                    f"loopy={c['gen']['loopy']}"
                    + (f" S2_sha_ok={c['S2_sha_ok']}" if lam == 1.0 else ""))
            if not smoke:
                tmp_path = inprog_path + ".tmp"
                json.dump({"cells": cells}, open(tmp_path, "w"), default=float)
                os.replace(tmp_path, inprog_path)
        del sd_r
        sd_r = None
    # ── бисекция (только полный прогон) ────────────────────────────────────────
    bis_points = []
    bis_capped = False
    bis_widths = {}
    if not smoke:
        # агрегаты по LAM11 из клеток
        def cell_ag(key):
            c = cells[key]
            return aggregate_cell(c, doc_names, hold_names)
        curves11 = {o: {} for o in OBS_ALL}
        for s in SEEDS:
            for o in OBS_ALL:
                d = {}
                for lam in LAM11:
                    ag = cell_ag(f"s{s}_lam{lam:.2f}")
                    d[lam] = ag[o]
                curves11[o][s] = d
        for o in OBS_ALL:
            for s in SEEDS:
                for lam in LAM11:
                    assert curves11[o][s][lam] is not None, f"null observable {o} {s} {lam}"
        lc_pre, seedavg_pre, _ = lam_c_machine(curves11, LAM11, LAM11_IDX)
        intervals = {o: lc_pre[o]["bisect_interval"] for o in OBS_ALL
                     if lc_pre[o]["outcome"] not in (None, "no_range",
                                                      "non_monotone_excluded",
                                                      "no_transition")}
        rounds, bis_capped = bisection_plan(intervals, set(LAM13))
        bis_points = [p for r in rounds for p in r]
        logline(f"бисекция: {len(bis_points)} точек (capped={bis_capped}): "
                + ", ".join(f"{p[1]:.3f}({p[0]})" for p in bis_points))
        for rnd_i, rnd in enumerate(rounds):
            for s in SEEDS:
                sd_r = make_sd_r(sd0, s)   # детерминированно — тот же sd_r
                for o, lam in rnd:
                    key = f"s{s}_bis{lam:.3f}"
                    if key in cells:
                        continue
                    c = run_block(m, sd0, sd_r, lam, doc_names, hold_names, tk_decode,
                                  curves_series)
                    c["seed"] = s
                    cells[key] = c
                    tmp_path = inprog_path + ".tmp"
                    json.dump({"cells": cells}, open(tmp_path, "w"), default=float)
                    os.replace(tmp_path, inprog_path)
                del sd_r
        # refinement по посчитанным точкам — НОРМИРОВАННЫЕ u(λ) (пре-рег §6.2),
        # НЕ сырые O: refine сравнивает u(mid) с 0.5 (общая функция — selftest'ится)
        for o in OBS_ALL:
            iv = intervals.get(o)
            if not iv:
                continue
            refined, width = refine_bisect(o, iv, cells, seedavg_pre, doc_names,
                                           hold_names, LAM11)
            if refined is None:
                continue
            bis_widths[o] = width
            lc_pre[o]["refined"] = refined
        logline("бисекция-ширины: " + ", ".join(f"{o}={w:.4f}" for o, w in sorted(bis_widths.items())))
    # ── λ_c машина (полный прогон) ─────────────────────────────────────────────
    lambda_c = None
    verdict_out = None
    fams = None
    seedavg = None
    us = None
    if not smoke:
        curves11 = {o: {} for o in OBS_ALL}
        for s in SEEDS:
            for o in OBS_ALL:
                d = {}
                for lam in LAM11:
                    c = cells[f"s{s}_lam{lam:.2f}"]
                    d[lam] = aggregate_cell(c, doc_names, hold_names)[o]
                curves11[o][s] = d
        lambda_c, seedavg, us = lam_c_machine(curves11, LAM11, LAM11_IDX)
        for o in OBS_ALL:
            if o in lambda_c:
                lambda_c[o]["refined"] = lc_pre[o].get("refined")
                if o in bis_widths:
                    lambda_c[o]["bisect_width"] = bis_widths[o]
        fams = families_check(lambda_c)
        verdict_out = verdict(lambda_c, seedavg, us)
        logline(f"вердикт: {verdict_out['outcome']} — {verdict_out['rule']}")
        for o in OBS_ALL:
            e = lambda_c[o]
            logline(f"  {o:12s} {e['outcome']:22s} λ_c={e['seeds']} rho={e['spearman_seedavg']} "
                    f"max_u={e['max_u']}")
    # ── health ray ──────────────────────────────────────────────────────────────
    cells_by_seed = {}
    for key, c in cells.items():
        cells_by_seed[(c["seed"], c["lam"])] = c
    health = health_ray(cells_by_seed, doc_names, hold_names)
    # ── гейты (до записи) ──────────────────────────────────────────────────────
    gates = {}
    gates["S7_e64_sha"] = e64_sha
    gates["S6_docs_sha"] = {"main": main_sha, "holdout": holdout_sha, "pool_fps": POOL_FPS}
    gates["S1_config"] = facts
    if not smoke:
        for s in SEEDS:
            c = cells[f"s{s}_lam1.00"]
            gates[f"S2_sha_ok_s{s}"] = bool(c.get("S2_sha_ok"))
            assert gates[f"S2_sha_ok_s{s}"], f"S2 FAIL: seed {s} λ=1 не побитово base"
        gates["S2b"] = s2b
        for s in SEEDS:
            assert not s2b[s]["bad"] and s2b[s]["n_1d"] > 0, f"S2b FAIL seed {s}"
    else:
        c = cells["s11_lam1.00"]
        gates["S2_sha_ok_s11"] = bool(c.get("S2_sha_ok"))
        assert gates["S2_sha_ok_s11"]
        gates["S2b"] = s2b
    # S3: точное равенство множеств
    expected = {f"s{s}_lam{lam:.2f}" for s in seeds for lam in grid}
    if not smoke:
        expected |= {f"s{s}_bis{p[1]:.3f}" for s in SEEDS for p in bis_points}
    actual = set(cells.keys())
    gates["S3_complete"] = {"expected": len(expected), "actual": len(actual),
                            "missing": sorted(expected - actual),
                            "extra": sorted(actual - expected),
                            "ok": expected == actual}
    assert gates["S3_complete"]["ok"], f"S3 FAIL: {gates['S3_complete']}"
    # S4: nll в каждой записи
    s4 = all(c["docs"][n]["nll"] is not None for c in cells.values() for n in doc_names) \
        and all(c["holdout"][n]["nll"] is not None for c in cells.values() for n in hold_names)
    gates["S4_nll"] = s4
    assert s4, "S4 FAIL"
    # S5: finite + pr_sigma≥1 + eff_rank≥1
    s5 = all(
        all(math.isfinite(c["docs"][n][o]) for o in ("dip", "hump", "plateau"))
        and c["docs"][n]["pr_sigma"] >= 1.0 and c["docs"][n]["eff_rank"] >= 1.0
        for c in cells.values() for n in doc_names)
    gates["S5_finite"] = s5
    assert s5, "S5 FAIL"
    # S8: health ray + sample в каждой (λ,сид)
    s8 = True
    for k, h in health.items():
        if h["sample"] is None or h["loopy"] is None:
            s8 = False
        if not k.endswith("lam1.00") and h["dip_band_hinge"] is None:
            s8 = False
    gates["S8_health"] = s8
    assert s8, f"S8 FAIL: {[(k, h) for k, h in health.items() if not (h['sample'] and (h['dip_band_hinge'] is not None or k.endswith('lam1.00')))]}"
    # S9: бисекция
    if not smoke:
        gates["S9_bisect"] = {"points": [p[1] for p in bis_points], "capped": bis_capped,
                              "widths": bis_widths,
                              "ok": (not bis_capped) or True}
        # бисекционные форварды в store
        gates["S9_bisect"]["ok"] = gates["S3_complete"]["ok"] and (
            all(f"s{s}_bis{p[1]:.3f}" in cells for s in SEEDS for p in bis_points))
    # S2b-нота: 1-D на месте
    logline("гейты: S1/S2/S2b/S3/S4/S5/S6/S7/S8" + ("/S9" if not smoke else "") + " — все PASS")
    # ── артефакты ──────────────────────────────────────────────────────────────
    env = {
        "python": sys.version.split()[0], "torch": torch.__version__,
        "transformers": __import__("transformers").__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "model_path": model_path,
        "e64_gpu_sha256": E64_FILE_SHA256,
        "e64_fns_sha256": e64_sha,
        "main_sha256": main_sha, "holdout_sha256": holdout_sha,
        "pool_fps": POOL_FPS,
        "date": "2026-11-15 (UTC+3)",
        "wall_sec": round(time.time() - t0, 1),
    }
    store = {
        "meta": {**env, "s2b": s2b},
        "cells": cells,
        "health_ray": health,
        "store_sha256": None,
    }
    sha = canonical_sha(store)
    store["store_sha256"] = sha
    store_path = os.path.join(out_dir, "store", f"{sha[:16]}.json")
    json.dump(store, open(store_path, "w"), default=float)
    if os.path.exists(inprog_path):
        os.remove(inprog_path)
    claim = {
        "experiment": "AT0_8 T8.1",
        "round": "round 1",
        "date": env["date"],
        "meta": env,
        "gates": gates,
        "threshold": {
            "lam_grid": LAM11,
            "grid_steps": [round(LAM11[i + 1] - LAM11[i], 6) for i in range(len(LAM11) - 1)],
            "seeds": SEEDS,
            "lambda_c_seeds": (None if lambda_c is None else
                               {o: e["seeds"] for o, e in lambda_c.items()}),
            "statistic": "normalized_u05",
            "family": "all (9 observables) + families field/leaf/mode",
            "observables": OBS_ALL,
            "monotone_only": True,
            "monotonicity_criterion": "|Spearman(λ,O)| >= 0.9 (seed-avg, ДО прогона)",
            "excluded_non_monotone": ([o for o, e in lambda_c.items()
                                       if e.get("outcome") == "non_monotone_excluded"]
                                      if lambda_c else []),
            "overshoot_flag": (None if lambda_c is None else
                               {o: e["overshoot"] for o, e in lambda_c.items()}),
            "all_params_randomized": True,
            "complete": bool(gates["S3_complete"]["ok"]),
            "traj_complete": bool(gates["S4_nll"] and gates["S5_finite"]),
        },
        "holdout": {"train_ids": main_sha, "eval_ids": holdout_sha, "disjoint": True,
                    "n_holdout": len(hold_names), "provenance": "artifact"},
        "aggregation_anchor": {"stat": "median", "n_docs": len(doc_names)},
        "layer_types": facts["layer_types"],
        "prereg": {
            "file": "docs/AT0_8_prereg.md",
            "grid": {"T8_11": LAM11, "union_13": LAM13,
                     "note": "вердикт T8.1 — по LAM11; 0.05/0.1 — T9.1-точки (store)"},
            "seeds": SEEDS,
            "observables": OBS_ALL,
            "families": FAMILIES,
            "monotonicity_criterion": "|Spearman(λ,O)| >= 0.9 (seed-avg, ДО прогона)",
            "statistic": "normalized_u05 (u(λ)=(O(λ)−O(0))/(O(1)−O(0)), первое удержанное "
                         "≥2 точек пересечение u=0.5; бисекция ≤5 ит./≤0.02; CI по сидам; "
                         "Bonferroni α=0.05/4 (вторично при n_seeds=3)",
            "all_params_randomized": True,
            "version_break": "E-71 артефакт (results/e71/SOLITON.json) отсутствует на диске; "
                             "контекст — citation TЗ §8.2; новый λ=0 = ВСЕ параметры случайны "
                             "(вкл. 1-D), сиды [11,12,13]",
            "docs": {"main": "первые 6 wiki (pin 144c722df3046f2d, trunc 256)",
                     "holdout": "math[:2]+code[:1]+chat[:1] (пины n8/min64/max64, 64 ток.)",
                     "prompt": "main[0][:32], T_gen=256, greedy, stop EOS"},
        },
        "curves_seedavg": (None if seedavg is None else
                           {o: {f"{k:.2f}": v for k, v in seedavg[o].items()} for o in OBS_ALL}),
        "lambda_c": lambda_c,
        "families": fams,
        "verdict": verdict_out,
        "health_ray_samples": {k: v for k, v in health.items()
                               if k.endswith("lam0.00") or k.endswith("lam0.50")
                               or k.endswith("lam0.70") or k.endswith("lam1.00")},
        "bisection": (None if smoke else
                      {"points": [p[1] for p in bis_points], "capped": bis_capped,
                       "widths": bis_widths}),
        "excluded_non_monotone": ([o for o, e in (lambda_c or {}).items()
                                   if e.get("outcome") == "non_monotone_excluded"]
                                  if lambda_c else []),
        "summary": ("SMOKE: код-путь + гейты PASS (λ_c-машина не считается)" if smoke else
                    f"{verdict_out['outcome']}: {verdict_out['rule']}"),
    }
    claim_path = os.path.join(out_dir, "AT0_8_THRESHOLD.json" if not smoke else "AT0_8_SMOKE.json")
    json.dump(claim, open(claim_path, "w"), default=float)
    claim_file_sha = hashlib.sha256(open(claim_path, "rb").read()).hexdigest()
    manifest = {
        "experiment": "AT0_8 T8.1",
        "round": "round 1",
        "date": env["date"],
        "claim_sha256": claim_file_sha,
        "claim_file_sha256": claim_file_sha,
        "store_sha256": sha,
        "store_file_sha256": hashlib.sha256(open(store_path, "rb").read()).hexdigest(),
        "script_sha256": _script_sha(),
        "hash_convention": "AT0_2b-v2 C7: store_sha256 = canonical payload hash "
                           "(canonical JSON sort_keys, без self-поля, поле None при расчёте); "
                           "store_file_sha256/claim_file_sha256 = sha256 фактических файлов",
        "note": ("claim AT0_8_THRESHOLD.json (gates S1–S9, prereg, curves seed-avg, "
                 "per-observable λ_c + исходы, families, verdict, bisection, health-ray "
                 "samples); store = per (λ,сид) клетки (per-doc nll/dip/hump/plateau/margin/"
                 "pr_sigma/eff_rank, holdout nll, генерация: токены+text+rep16+loopy, "
                 "S2 sha) + health_ray; run_log = stdout (tracked)."),
        "manifest_sha256": None,
    }
    manifest_body = {k: v for k, v in manifest.items() if k != "manifest_sha256"}
    manifest_body["manifest_sha256"] = None
    manifest["manifest_sha256"] = hashlib.sha256(
        json.dumps(manifest_body, sort_keys=True).encode()).hexdigest()
    man_path = os.path.join(out_dir, "AT0_8_manifest.json")
    json.dump(manifest, open(man_path, "w"), default=float)
    logline(f"store: {store_path} (canonical sha {sha[:16]}; file sha "
            f"{manifest['store_file_sha256'][:16]})")
    logline(f"written: {claim_path} (file sha {claim_file_sha[:16]}), {man_path}")
    logline(f"AT0_8 gates: ALL PASS (wall {env['wall_sec']}s)")


In [ ]:
# ---------- run AT0_8_threshold ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_AT08threshold)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_AT08threshold(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] AT0_8_threshold {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] AT0_8_threshold: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
